# Planner + Caller — concise English plans — compiled Unsloth
Select T4 x2, Internet on, Run All. No job is launched by saving this notebook.
Train Planner and Caller only, independently from original Qwen. No Summarizer or Global training by default. English prompts and concise field-source rationales; tool contracts are TypeScript, actions remain JSON. The split remains 1300 train / 245 validation episodes.
This notebook starts full finetuning directly: no automatic smoke training, forced pause/resume, or synthetic GPU preflight. Package consistency and input checks remain. Unsloth compilation is enabled; its first compilation can take time and GPU speedup must be measured.
Planner decides the next tool, constraints and termination; Caller resolves argument values from the request, history and past observations. Labels name parameter sources without serializing the target arguments. Deterministic labels require semantic review, especially complex conditions and repeated same-tool objectives.
Planner and Caller: 1 epoch each. Train/validation microbatch 4 per GPU, effective train batch 8. Unsloth QLoRA rank16, cosine/warmup, patience3; loss chunks256, cached tokenization.
Checkpoint every25 updates and after evaluation: adapter, optimizer, LR scheduler, scaler, per-rank RNG and batch cursor. At 10 hours request a cooperative save/pause. Hard kill can lose work since the last complete checkpoint. Kaggle working storage is not permanent: preserve outputs and attach them as a private input for a later session. RESUME_DIR must point to the previous whole run directory including prepared/, full/ and checkpoints/best.
This notebook only prepares and finetunes; no ASR, workflow/public-test inference or automatic submission. Validation loss remains teacher-forced and is not a replacement for Planner free-generation evaluation. No validation-specific prompt exceptions are used.


In [ ]:
from pathlib import Path
import os,sys,json,hashlib,subprocess,zipfile,shutil,time
PROFILE='full' # Full training only; no automatic smoke test
TRAIN_MODE='local' # 'local', 'global', or 'global_to_local'
LOCAL_INIT='base' # local mode: 'base' or 'global'
GLOBAL_ADAPTER=None # completed Global adapter directory; only for local + LOCAL_INIT='global'
LOCAL_STAGES=('planner','caller') # optionally select a subset
RUN_NAME='planner_caller_concise_compiled_v1'
assert TRAIN_MODE in ('local','global','global_to_local') and LOCAL_INIT in ('base','global')
if TRAIN_MODE=='local':assert (GLOBAL_ADAPTER is not None)==(LOCAL_INIT=='global')
else:assert GLOBAL_ADAPTER is None,'GLOBAL_ADAPTER only applies to local mode'
MODE_TAG=TRAIN_MODE+'_'+LOCAL_INIT if TRAIN_MODE=='local' else TRAIN_MODE
RESUME_DIR=None # Mounted previous whole run directory, not just a single adapter
WORK=Path('/kaggle/working/planner_caller_compiled')/RUN_NAME/MODE_TAG
WORK.mkdir(parents=True,exist_ok=True)
if RESUME_DIR is not None:shutil.copytree(Path(RESUME_DIR),WORK,dirs_exist_ok=True)
SOURCE=Path('/kaggle/working/planner_caller_source')
MODEL='Qwen/Qwen3-4B-Instruct-2507';REVISION='cdbee75f17c01a7cc42f958dc650907174af0554'
EPOCHS={'global':1,'planner':1,'caller':1,'summarizer':2}
DEADLINE=time.time()+10*3600
assert PROFILE=='full'
assert set(LOCAL_STAGES) <= {'planner','caller'} and LOCAL_STAGES
if not SOURCE.exists():subprocess.run(['git','clone','--filter=blob:none','--no-checkout','https://github.com/tnisl/baseline.git',str(SOURCE)],check=True)
subprocess.run(['git','-C',str(SOURCE),'fetch','origin','4894d331692293565635454a8c686b7c18314a04'],check=True)
subprocess.run(['git','-C',str(SOURCE),'checkout','--detach','FETCH_HEAD'],check=True)


In [ ]:
PAYLOAD={'requirements_train_2gpu.lock': 'accelerate==1.10.1\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   peft\n    #   trl\n    #   unsloth\n    #   unsloth-zoo\nagent-detector==2.0.0\n    # via fastapi-cloud-cli\naiohappyeyeballs==2.7.1\n    # via aiohttp\naiohttp==3.14.5\n    # via\n    #   fsspec\n    #   vllm\naiosignal==1.4.0\n    # via aiohttp\nannotated-doc==0.0.5\n    # via\n    #   fastapi\n    #   typer\nannotated-types==0.8.0\n    # via pydantic\nanyio==4.16.0\n    # via\n    #   httpx\n    #   httpx2\n    #   openai\n    #   starlette\n    #   watchfiles\nastor==0.8.1\n    # via depyf\nattrs==26.1.0\n    # via\n    #   aiohttp\n    #   jsonschema\n    #   referencing\nbitsandbytes==0.47.0\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   unsloth\nblake3==1.0.11\n    # via vllm\ncachetools==7.2.8\n    # via vllm\ncbor2==6.2.0.post2\n    # via vllm\ncertifi==2026.10.23\n    # via\n    #   httpcore\n    #   httpx\n    #   requests\n    #   sentry-sdk\ncffi==2.1.1\n    # via soundfile\ncharset-normalizer==3.5.2\n    # via requests\nclick==8.5.0\n    # via\n    #   ray\n    #   rich-toolkit\n    #   uvicorn\ncloudpickle==3.1.2\n    # via vllm\ncompressed-tensors==0.11.0\n    # via vllm\ncuda-pathfinder==1.8.5\n    # via cupy-cuda12x\ncupy-cuda12x==14.2.0\n    # via ray\ncut-cross-entropy==25.1.1\n    # via unsloth-zoo\ndatasets==3.6.0\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   trl\n    #   unsloth\n    #   unsloth-zoo\ndepyf==0.19.0\n    # via vllm\ndetect-installer==0.2.1\n    # via fastapi-cloud-cli\ndiffusers==0.35.1\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   unsloth\ndill==0.3.8\n    # via\n    #   datasets\n    #   depyf\n    #   multiprocess\ndiskcache==5.6.3\n    # via vllm\ndnspython==2.9.1\n    # via email-validator\ndocstring-parser==0.18.0\n    # via tyro\neinops==0.8.2\n    # via vllm\nemail-validator==2.3.0\n    # via\n    #   fastapi\n    #   pydantic\nfastapi==0.162.0\n    # via vllm\nfastapi-cli==0.0.32\n    # via fastapi\nfastapi-cloud-cli==0.26.0\n    # via fastapi-cli\nfastar==0.12.0\n    # via\n    #   fastapi\n    #   fastapi-cloud-cli\nfilelock==4.2.7\n    # via\n    #   datasets\n    #   diffusers\n    #   huggingface-hub\n    #   ray\n    #   torch\n    #   transformers\n    #   unsloth-zoo\n    #   vllm\nfrozendict==2.5.1\n    # via compressed-tensors\nfrozenlist==1.8.0\n    # via\n    #   aiohttp\n    #   aiosignal\nfsspec==2025.3.0\n    # via\n    #   datasets\n    #   huggingface-hub\n    #   torch\ngguf==0.19.0\n    # via vllm\ngoogleapis-common-protos==1.76.0\n    # via opentelemetry-exporter-otlp-proto-http\nh11==0.16.0\n    # via\n    #   httpcore\n    #   httpcore2\n    #   uvicorn\nhf-transfer==0.1.9\n    # via\n    #   unsloth\n    #   unsloth-zoo\nhf-xet==1.8.0\n    # via huggingface-hub\nhttpcore==1.0.9\n    # via httpx\nhttpcore2==2.16.0\n    # via httpx2\nhttptools==0.9.0\n    # via uvicorn\nhttpx==0.28.1\n    # via\n    #   fastapi\n    #   fastapi-cloud-cli\nhttpx2==2.16.0\n    # via\n    #   fastapi\n    #   openai\nhuggingface-hub==0.35.3\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   accelerate\n    #   datasets\n    #   diffusers\n    #   peft\n    #   tokenizers\n    #   transformers\n    #   unsloth\n    #   unsloth-zoo\nidna==3.20\n    # via\n    #   anyio\n    #   email-validator\n    #   httpx\n    #   httpx2\n    #   requests\n    #   yarl\nimportlib-metadata==10.1.0\n    # via diffusers\ninteregular==0.3.3\n    # via lm-format-enforcer\njinja2==3.1.6\n    # via\n    #   fastapi\n    #   torch\njiter==0.18.0\n    # via openai\njsonschema==4.26.0\n    # via\n    #   mistral-common\n    #   ray\njsonschema-specifications==2025.9.1\n    # via jsonschema\nlark==1.2.2\n    # via vllm\nllguidance==0.7.30\n    # via vllm\nllvmlite==0.44.0\n    # via numba\nlm-format-enforcer==0.11.3\n    # via vllm\nmarkdown-it-py==4.2.0\n    # via rich\nmarkupsafe==3.0.4\n    # via jinja2\nmdurl==0.1.2\n    # via markdown-it-py\nmistral-common==1.13.1\n    # via vllm\nmpmath==1.3.0\n    # via sympy\nmsgpack==1.2.4\n    # via ray\nmsgspec==0.22.0\n    # via\n    #   unsloth-zoo\n    #   vllm\nmultidict==7.2.1\n    # via\n    #   aiohttp\n    #   yarl\nmultiprocess==0.70.16\n    # via datasets\nnetworkx==3.7\n    # via torch\nninja==1.13.2\n    # via\n    #   vllm\n    #   xgrammar\nnumba==0.61.2\n    # via vllm\nnumpy==2.2.6\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   accelerate\n    #   bitsandbytes\n    #   cupy-cuda12x\n    #   datasets\n    #   diffusers\n    #   gguf\n    #   mistral-common\n    #   numba\n    #   opencv-python-headless\n    #   pandas\n    #   peft\n    #   scipy\n    #   soundfile\n    #   soxr\n    #   torchvision\n    #   transformers\n    #   unsloth\n    #   unsloth-zoo\n    #   vllm\n    #   xformers\n    #   xgrammar\nnvidia-cublas-cu12==12.8.4.1\n    # via\n    #   nvidia-cudnn-cu12\n    #   nvidia-cusolver-cu12\n    #   torch\nnvidia-cuda-cupti-cu12==12.8.90\n    # via torch\nnvidia-cuda-nvrtc-cu12==12.8.93\n    # via torch\nnvidia-cuda-runtime-cu12==12.8.90\n    # via torch\nnvidia-cudnn-cu12==9.10.2.21\n    # via torch\nnvidia-cufft-cu12==11.3.3.83\n    # via torch\nnvidia-cufile-cu12==1.13.1.3\n    # via torch\nnvidia-curand-cu12==10.3.9.90\n    # via torch\nnvidia-cusolver-cu12==11.7.3.90\n    # via torch\nnvidia-cusparse-cu12==12.5.8.93\n    # via\n    #   nvidia-cusolver-cu12\n    #   torch\nnvidia-cusparselt-cu12==0.7.1\n    # via torch\nnvidia-nccl-cu12==2.27.3\n    # via torch\nnvidia-nvjitlink-cu12==12.8.93\n    # via\n    #   nvidia-cufft-cu12\n    #   nvidia-cusolver-cu12\n    #   nvidia-cusparse-cu12\n    #   torch\nnvidia-nvtx-cu12==12.8.90\n    # via torch\nopenai==3.42.1\n    # via vllm\nopenai-harmony==0.0.8\n    # via vllm\nopencv-python-headless==5.0.0.93\n    # via\n    #   mistral-common\n    #   vllm\nopentelemetry-api==1.45.2\n    # via\n    #   fastapi\n    #   opentelemetry-exporter-http-transport\n    #   opentelemetry-exporter-otlp-proto-http\n    #   opentelemetry-sdk\n    #   opentelemetry-semantic-conventions\nopentelemetry-exporter-http-transport==0.66b2\n    # via opentelemetry-exporter-otlp-proto-http\nopentelemetry-exporter-otlp-common==0.66b2\n    # via opentelemetry-exporter-otlp-proto-http\nopentelemetry-exporter-otlp-proto-common==1.45.2\n    # via opentelemetry-exporter-otlp-proto-http\nopentelemetry-exporter-otlp-proto-http==1.45.2\n    # via fastapi\nopentelemetry-proto==1.45.2\n    # via\n    #   opentelemetry-exporter-otlp-proto-common\n    #   opentelemetry-exporter-otlp-proto-http\nopentelemetry-sdk==1.45.2\n    # via\n    #   fastapi\n    #   opentelemetry-exporter-otlp-common\n    #   opentelemetry-exporter-otlp-proto-http\nopentelemetry-semantic-conventions==0.66b2\n    # via opentelemetry-sdk\noutlines-core==0.2.11\n    # via vllm\npackaging==26.3\n    # via\n    #   accelerate\n    #   datasets\n    #   huggingface-hub\n    #   lm-format-enforcer\n    #   peft\n    #   ray\n    #   transformers\n    #   unsloth\n    #   unsloth-zoo\n    #   wheel\npandas==3.1.0\n    # via datasets\npartial-json-parser==0.2.1.1.post7\n    # via vllm\npeft==0.17.1\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   unsloth\n    #   unsloth-zoo\npillow==13.0.0\n    # via\n    #   diffusers\n    #   mistral-common\n    #   torchvision\n    #   unsloth-zoo\n    #   vllm\nprometheus-client==0.26.0\n    # via\n    #   prometheus-fastapi-instrumentator\n    #   vllm\nprometheus-fastapi-instrumentator==8.1.0\n    # via vllm\npropcache==0.5.4\n    # via\n    #   aiohttp\n    #   yarl\nprotobuf==7.36.2\n    # via\n    #   googleapis-common-protos\n    #   opentelemetry-proto\n    #   ray\n    #   unsloth\n    #   unsloth-zoo\n    #   vllm\npsutil==7.2.2\n    # via\n    #   accelerate\n    #   peft\n    #   unsloth\n    #   unsloth-zoo\n    #   vllm\npy-cpuinfo==9.0.0\n    # via vllm\npyarrow==26.0.0\n    # via datasets\npybase64==1.5.1\n    # via vllm\npycountry==26.2.16\n    # via pydantic-extra-types\npycparser==3.12\n    # via cffi\npydantic==2.14.2\n    # via\n    #   compressed-tensors\n    #   fastapi\n    #   fastapi-cloud-cli\n    #   lm-format-enforcer\n    #   mistral-common\n    #   openai\n    #   openai-harmony\n    #   pydantic-extra-types\n    #   pydantic-settings\n    #   vllm\n    #   xgrammar\npydantic-core==2.50.2\n    # via pydantic\npydantic-extra-types==2.11.1\n    # via\n    #   fastapi\n    #   mistral-common\npydantic-settings==2.16.2\n    # via fastapi\npygments==2.21.0\n    # via rich\npython-dateutil==4.0.0\n    # via pandas\npython-dotenv==1.2.4\n    # via\n    #   pydantic-settings\n    #   uvicorn\npython-json-logger==4.2.0\n    # via vllm\npython-multipart==0.0.33\n    # via fastapi\npyyaml==6.0.3\n    # via\n    #   accelerate\n    #   datasets\n    #   gguf\n    #   huggingface-hub\n    #   lm-format-enforcer\n    #   peft\n    #   ray\n    #   transformers\n    #   uvicorn\n    #   vllm\npyzmq==27.2.0\n    # via vllm\nray==2.59.0\n    # via vllm\nreferencing==0.37.0\n    # via\n    #   jsonschema\n    #   jsonschema-specifications\nregex==2026.9.29\n    # via\n    #   diffusers\n    #   tiktoken\n    #   transformers\n    #   unsloth-zoo\n    #   vllm\nrequests==2.34.2\n    # via\n    #   datasets\n    #   diffusers\n    #   gguf\n    #   huggingface-hub\n    #   mistral-common\n    #   opentelemetry-exporter-http-transport\n    #   opentelemetry-exporter-otlp-proto-http\n    #   ray\n    #   tiktoken\n    #   transformers\n    #   vllm\nrich==15.0.0\n    # via\n    #   rich-toolkit\n    #   typer\nrich-toolkit==0.20.6\n    # via\n    #   fastapi-cli\n    #   fastapi-cloud-cli\nrignore==0.8.1\n    # via fastapi-cloud-cli\nrpds-py==2026.9.1\n    # via\n    #   jsonschema\n    #   referencing\nsafetensors==0.8.0\n    # via\n    #   accelerate\n    #   diffusers\n    #   peft\n    #   transformers\nscipy==1.18.1\n    # via vllm\nsentencepiece==0.2.2\n    # via\n    #   unsloth\n    #   unsloth-zoo\n    #   vllm\nsentry-sdk==2.73.0\n    # via fastapi-cloud-cli\nsetproctitle==1.3.8\n    # via vllm\nsetuptools==79.0.1\n    # via\n    #   torch\n    #   triton\n    #   vllm\nshellingham==1.5.4\n    # via typer\nsix==1.17.0\n    # via vllm\nsniffio==1.3.1\n    # via openai\nsoundfile==0.14.0\n    # via mistral-common\nsoxr==1.1.0\n    # via mistral-common\nstarlette==1.10.0\n    # via\n    #   fastapi\n    #   prometheus-fastapi-instrumentator\nsympy==1.14.0\n    # via torch\ntiktoken==0.14.0\n    # via\n    #   mistral-common\n    #   vllm\ntokenizers==0.22.2\n    # via\n    #   transformers\n    #   vllm\ntorch==2.8.0\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   accelerate\n    #   bitsandbytes\n    #   compressed-tensors\n    #   cut-cross-entropy\n    #   peft\n    #   torchaudio\n    #   torchvision\n    #   unsloth\n    #   unsloth-zoo\n    #   vllm\n    #   xformers\n    #   xgrammar\ntorchao==0.13.0\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   unsloth-zoo\ntorchaudio==2.8.0\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   vllm\ntorchvision==0.23.0\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   unsloth\n    #   vllm\ntqdm==4.70.1\n    # via\n    #   datasets\n    #   gguf\n    #   huggingface-hub\n    #   peft\n    #   transformers\n    #   unsloth\n    #   unsloth-zoo\n    #   vllm\ntransformers==4.56.2\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   compressed-tensors\n    #   peft\n    #   trl\n    #   unsloth\n    #   unsloth-zoo\n    #   vllm\n    #   xgrammar\ntriton==3.4.0\n    # via\n    #   cut-cross-entropy\n    #   torch\n    #   unsloth\n    #   unsloth-zoo\n    #   xgrammar\ntrl==0.23.0\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   unsloth\n    #   unsloth-zoo\ntruststore==0.10.5\n    # via\n    #   httpcore2\n    #   httpx2\ntypeguard==4.6.0\n    # via tyro\ntyper==0.27.3\n    # via\n    #   fastapi-cli\n    #   fastapi-cloud-cli\ntyping-extensions==4.16.0\n    # via\n    #   aiohttp\n    #   aiosignal\n    #   anyio\n    #   fastapi\n    #   httpx2\n    #   huggingface-hub\n    #   openai\n    #   opentelemetry-api\n    #   opentelemetry-exporter-otlp-proto-http\n    #   opentelemetry-sdk\n    #   opentelemetry-semantic-conventions\n    #   pydantic\n    #   pydantic-core\n    #   pydantic-extra-types\n    #   pydantic-settings\n    #   referencing\n    #   rich-toolkit\n    #   soundfile\n    #   starlette\n    #   torch\n    #   typeguard\n    #   typing-inspection\n    #   tyro\n    #   unsloth-zoo\n    #   vllm\n    #   xgrammar\ntyping-inspection==0.4.4\n    # via\n    #   fastapi\n    #   pydantic\n    #   pydantic-settings\ntyro==1.0.16\n    # via\n    #   unsloth\n    #   unsloth-zoo\ntzdata==2026.6\n    # via python-dateutil\nunsloth==2025.10.9\n    # via -r /tmp/lifemate_stack_sources/requirements.in\nunsloth-zoo==2025.10.10\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   unsloth\nurllib3==2.8.0\n    # via\n    #   requests\n    #   sentry-sdk\nuvicorn==0.54.0\n    # via\n    #   fastapi\n    #   fastapi-cli\n    #   fastapi-cloud-cli\nuvloop==0.23.0\n    # via uvicorn\nvllm==0.10.2\n    # via -r /tmp/lifemate_stack_sources/requirements.in\nwatchfiles==1.3.0\n    # via\n    #   uvicorn\n    #   vllm\nwebsockets==17.4\n    # via uvicorn\nwheel==0.48.0\n    # via\n    #   unsloth\n    #   unsloth-zoo\nwrapt==1.17.3\n    # via -r /tmp/lifemate_stack_sources/requirements.in\nxformers==0.0.32.post1\n    # via\n    #   -r /tmp/lifemate_stack_sources/requirements.in\n    #   unsloth\n    #   vllm\nxgrammar==0.1.23\n    # via vllm\nxxhash==4.0.2\n    # via datasets\nyarl==1.25.1\n    # via aiohttp\nzipp==4.1.2\n    # via importlib-metadata\n', 'split.txt': 'train_0001\ntrain_0004\ntrain_0015\ntrain_0016\ntrain_0017\ntrain_0031\ntrain_0063\ntrain_0066\ntrain_0075\ntrain_0080\ntrain_0088\ntrain_0090\ntrain_0095\ntrain_0106\ntrain_0110\ntrain_0117\ntrain_0130\ntrain_0133\ntrain_0141\ntrain_0144\ntrain_0145\ntrain_0147\ntrain_0162\ntrain_0163\ntrain_0164\ntrain_0169\ntrain_0172\ntrain_0181\ntrain_0194\ntrain_0196\ntrain_0203\ntrain_0207\ntrain_0214\ntrain_0216\ntrain_0219\ntrain_0225\ntrain_0228\ntrain_0231\ntrain_0233\ntrain_0236\ntrain_0240\ntrain_0255\ntrain_0278\ntrain_0285\ntrain_0304\ntrain_0313\ntrain_0316\ntrain_0321\ntrain_0330\ntrain_0344\ntrain_0345\ntrain_0346\ntrain_0350\ntrain_0354\ntrain_0357\ntrain_0378\ntrain_0381\ntrain_0395\ntrain_0405\ntrain_0422\ntrain_0435\ntrain_0442\ntrain_0453\ntrain_0457\ntrain_0468\ntrain_0469\ntrain_0476\ntrain_0480\ntrain_0481\ntrain_0486\ntrain_0497\ntrain_0500\ntrain_0505\ntrain_0508\ntrain_0518\ntrain_0536\ntrain_0538\ntrain_0550\ntrain_0554\ntrain_0560\ntrain_0563\ntrain_0572\ntrain_0577\ntrain_0596\ntrain_0598\ntrain_0618\ntrain_0621\ntrain_0624\ntrain_0625\ntrain_0628\ntrain_0664\ntrain_0689\ntrain_0694\ntrain_0719\ntrain_0729\ntrain_0752\ntrain_0764\ntrain_0770\ntrain_0772\ntrain_0777\ntrain_0779\ntrain_0787\ntrain_0790\ntrain_0800\ntrain_0805\ntrain_0814\ntrain_0815\ntrain_0819\ntrain_0820\ntrain_0834\ntrain_0857\ntrain_0864\ntrain_0867\ntrain_0870\ntrain_0875\ntrain_0923\ntrain_0933\ntrain_0935\ntrain_0943\ntrain_0945\ntrain_0947\ntrain_0948\ntrain_0953\ntrain_0958\ntrain_0959\ntrain_0960\ntrain_0977\ntrain_0978\ntrain_0991\ntrain_1000\ntrain_1014\ntrain_1016\ntrain_1019\ntrain_1021\ntrain_1049\ntrain_1060\ntrain_1062\ntrain_1070\ntrain_1081\ntrain_1089\ntrain_1091\ntrain_1096\ntrain_1097\ntrain_1109\ntrain_1117\ntrain_1118\ntrain_1123\ntrain_1124\ntrain_1128\ntrain_1137\ntrain_1147\ntrain_1148\ntrain_1155\ntrain_1160\ntrain_1163\ntrain_1175\ntrain_1180\ntrain_1188\ntrain_1201\ntrain_1206\ntrain_1209\ntrain_1215\ntrain_1216\ntrain_1233\ntrain_1234\ntrain_1246\ntrain_1248\ntrain_1249\ntrain_1253\ntrain_1265\ntrain_1278\ntrain_1289\ntrain_1295\ntrain_1300\ntrain_1305\ntrain_1310\ntrain_1311\ntrain_1317\ntrain_1325\ntrain_1346\ntrain_1351\ntrain_1352\ntrain_1353\ntrain_1355\ntrain_1356\ntrain_1363\ntrain_1373\ntrain_1379\ntrain_1380\ntrain_1384\ntrain_1399\ntrain_1405\ntrain_1408\ntrain_1432\ntrain_1433\ntrain_1437\ntrain_1440\ntrain_1448\ntrain_1458\ntrain_1460\ntrain_1471\ntrain_1472\ntrain_1474\ntrain_1475\ntrain_1476\ntrain_1478\ntrain_1486\ntrain_1491\ntrain_1497\ntrain_1506\ntrain_1512\ntrain_1519\ntrain_1526\ntrain_1528\ntrain_1533\ntrain_1540\ntrain_1542\ntrain_1544\ntrain_1546\ntrain_1556\ntrain_1557\ntrain_1560\ntrain_1571\ntrain_1572\ntrain_1589\ntrain_1598\ntrain_1606\ntrain_1608\ntrain_1615\ntrain_1617\ntrain_1623\ntrain_1653\ntrain_1654\ntrain_1658\ntrain_1661\ntrain_1668\ntrain_1697\ntrain_1702\ntrain_1703\ntrain_1705\ntrain_1707\ntrain_1712\ntrain_1713\ntrain_1715\ntrain_1746\n', 'split_manifest.json': '{\n  "seed": "english-split-20261104",\n  "train": 1300,\n  "validation": 245,\n  "original_valid_retained": 100,\n  "additional_valid": 145,\n  "strata": "first-tool app, capped call count, conditional keyword; proportional largest remainder",\n  "split_sha256": "ce713fbd99bf0bad81420935477f545af1b97fa96e5b606c6ea9377e465b7427",\n  "requires_new_global": true,\n  "note": "ID-disjoint split. Semantic paraphrase groups have not been manually audited; old 100 are a development set."\n}\n', 'train_ids.txt': 'train_0002\ntrain_0003\ntrain_0005\ntrain_0006\ntrain_0007\ntrain_0008\ntrain_0009\ntrain_0011\ntrain_0012\ntrain_0013\ntrain_0014\ntrain_0018\ntrain_0019\ntrain_0020\ntrain_0025\ntrain_0026\ntrain_0027\ntrain_0028\ntrain_0029\ntrain_0030\ntrain_0032\ntrain_0033\ntrain_0034\ntrain_0035\ntrain_0036\ntrain_0037\ntrain_0038\ntrain_0039\ntrain_0040\ntrain_0041\ntrain_0042\ntrain_0043\ntrain_0045\ntrain_0046\ntrain_0047\ntrain_0048\ntrain_0049\ntrain_0050\ntrain_0052\ntrain_0055\ntrain_0057\ntrain_0059\ntrain_0060\ntrain_0061\ntrain_0062\ntrain_0064\ntrain_0065\ntrain_0068\ntrain_0069\ntrain_0070\ntrain_0071\ntrain_0072\ntrain_0074\ntrain_0076\ntrain_0078\ntrain_0079\ntrain_0081\ntrain_0082\ntrain_0083\ntrain_0084\ntrain_0085\ntrain_0086\ntrain_0087\ntrain_0089\ntrain_0091\ntrain_0092\ntrain_0093\ntrain_0094\ntrain_0096\ntrain_0097\ntrain_0098\ntrain_0100\ntrain_0103\ntrain_0104\ntrain_0105\ntrain_0107\ntrain_0108\ntrain_0109\ntrain_0111\ntrain_0112\ntrain_0113\ntrain_0115\ntrain_0116\ntrain_0119\ntrain_0121\ntrain_0122\ntrain_0123\ntrain_0124\ntrain_0125\ntrain_0126\ntrain_0127\ntrain_0128\ntrain_0129\ntrain_0131\ntrain_0132\ntrain_0134\ntrain_0135\ntrain_0136\ntrain_0137\ntrain_0138\ntrain_0139\ntrain_0140\ntrain_0142\ntrain_0143\ntrain_0146\ntrain_0148\ntrain_0149\ntrain_0150\ntrain_0151\ntrain_0152\ntrain_0153\ntrain_0154\ntrain_0155\ntrain_0156\ntrain_0157\ntrain_0158\ntrain_0159\ntrain_0161\ntrain_0165\ntrain_0166\ntrain_0168\ntrain_0170\ntrain_0171\ntrain_0173\ntrain_0174\ntrain_0176\ntrain_0177\ntrain_0178\ntrain_0179\ntrain_0180\ntrain_0182\ntrain_0183\ntrain_0184\ntrain_0185\ntrain_0186\ntrain_0187\ntrain_0188\ntrain_0189\ntrain_0190\ntrain_0191\ntrain_0192\ntrain_0193\ntrain_0195\ntrain_0197\ntrain_0199\ntrain_0200\ntrain_0201\ntrain_0202\ntrain_0204\ntrain_0205\ntrain_0206\ntrain_0208\ntrain_0209\ntrain_0210\ntrain_0211\ntrain_0212\ntrain_0213\ntrain_0215\ntrain_0217\ntrain_0220\ntrain_0221\ntrain_0222\ntrain_0224\ntrain_0226\ntrain_0227\ntrain_0229\ntrain_0230\ntrain_0232\ntrain_0234\ntrain_0235\ntrain_0237\ntrain_0238\ntrain_0239\ntrain_0241\ntrain_0242\ntrain_0243\ntrain_0244\ntrain_0245\ntrain_0246\ntrain_0247\ntrain_0248\ntrain_0249\ntrain_0250\ntrain_0251\ntrain_0252\ntrain_0253\ntrain_0254\ntrain_0256\ntrain_0257\ntrain_0258\ntrain_0259\ntrain_0260\ntrain_0261\ntrain_0262\ntrain_0263\ntrain_0264\ntrain_0265\ntrain_0266\ntrain_0267\ntrain_0268\ntrain_0269\ntrain_0270\ntrain_0271\ntrain_0272\ntrain_0273\ntrain_0274\ntrain_0275\ntrain_0276\ntrain_0277\ntrain_0279\ntrain_0280\ntrain_0281\ntrain_0282\ntrain_0283\ntrain_0284\ntrain_0286\ntrain_0287\ntrain_0288\ntrain_0289\ntrain_0290\ntrain_0291\ntrain_0292\ntrain_0293\ntrain_0294\ntrain_0295\ntrain_0296\ntrain_0297\ntrain_0298\ntrain_0299\ntrain_0300\ntrain_0301\ntrain_0302\ntrain_0303\ntrain_0306\ntrain_0307\ntrain_0308\ntrain_0309\ntrain_0310\ntrain_0311\ntrain_0312\ntrain_0314\ntrain_0315\ntrain_0317\ntrain_0318\ntrain_0319\ntrain_0320\ntrain_0322\ntrain_0323\ntrain_0324\ntrain_0325\ntrain_0326\ntrain_0327\ntrain_0328\ntrain_0329\ntrain_0331\ntrain_0332\ntrain_0333\ntrain_0334\ntrain_0335\ntrain_0336\ntrain_0337\ntrain_0338\ntrain_0339\ntrain_0340\ntrain_0341\ntrain_0342\ntrain_0343\ntrain_0347\ntrain_0348\ntrain_0349\ntrain_0351\ntrain_0352\ntrain_0353\ntrain_0355\ntrain_0356\ntrain_0358\ntrain_0359\ntrain_0360\ntrain_0361\ntrain_0362\ntrain_0363\ntrain_0364\ntrain_0365\ntrain_0366\ntrain_0367\ntrain_0368\ntrain_0369\ntrain_0370\ntrain_0371\ntrain_0372\ntrain_0373\ntrain_0374\ntrain_0375\ntrain_0376\ntrain_0377\ntrain_0379\ntrain_0380\ntrain_0382\ntrain_0383\ntrain_0384\ntrain_0385\ntrain_0386\ntrain_0387\ntrain_0388\ntrain_0389\ntrain_0390\ntrain_0391\ntrain_0392\ntrain_0393\ntrain_0394\ntrain_0396\ntrain_0397\ntrain_0398\ntrain_0399\ntrain_0400\ntrain_0401\ntrain_0402\ntrain_0403\ntrain_0404\ntrain_0406\ntrain_0407\ntrain_0408\ntrain_0409\ntrain_0410\ntrain_0411\ntrain_0412\ntrain_0413\ntrain_0414\ntrain_0415\ntrain_0416\ntrain_0417\ntrain_0418\ntrain_0419\ntrain_0420\ntrain_0421\ntrain_0423\ntrain_0424\ntrain_0425\ntrain_0426\ntrain_0427\ntrain_0428\ntrain_0429\ntrain_0430\ntrain_0431\ntrain_0432\ntrain_0433\ntrain_0434\ntrain_0436\ntrain_0437\ntrain_0439\ntrain_0440\ntrain_0441\ntrain_0443\ntrain_0444\ntrain_0445\ntrain_0446\ntrain_0447\ntrain_0448\ntrain_0449\ntrain_0450\ntrain_0451\ntrain_0452\ntrain_0454\ntrain_0455\ntrain_0456\ntrain_0458\ntrain_0459\ntrain_0460\ntrain_0461\ntrain_0462\ntrain_0463\ntrain_0464\ntrain_0465\ntrain_0466\ntrain_0467\ntrain_0470\ntrain_0471\ntrain_0472\ntrain_0473\ntrain_0474\ntrain_0475\ntrain_0477\ntrain_0478\ntrain_0479\ntrain_0482\ntrain_0483\ntrain_0484\ntrain_0485\ntrain_0488\ntrain_0490\ntrain_0491\ntrain_0493\ntrain_0494\ntrain_0495\ntrain_0496\ntrain_0498\ntrain_0499\ntrain_0501\ntrain_0502\ntrain_0503\ntrain_0504\ntrain_0506\ntrain_0507\ntrain_0509\ntrain_0510\ntrain_0511\ntrain_0512\ntrain_0513\ntrain_0514\ntrain_0515\ntrain_0516\ntrain_0517\ntrain_0520\ntrain_0523\ntrain_0526\ntrain_0527\ntrain_0528\ntrain_0529\ntrain_0530\ntrain_0531\ntrain_0532\ntrain_0533\ntrain_0534\ntrain_0535\ntrain_0537\ntrain_0539\ntrain_0540\ntrain_0541\ntrain_0543\ntrain_0544\ntrain_0545\ntrain_0546\ntrain_0548\ntrain_0551\ntrain_0556\ntrain_0557\ntrain_0558\ntrain_0559\ntrain_0562\ntrain_0565\ntrain_0567\ntrain_0568\ntrain_0570\ntrain_0571\ntrain_0573\ntrain_0574\ntrain_0575\ntrain_0578\ntrain_0579\ntrain_0581\ntrain_0583\ntrain_0584\ntrain_0587\ntrain_0588\ntrain_0589\ntrain_0593\ntrain_0594\ntrain_0595\ntrain_0597\ntrain_0599\ntrain_0600\ntrain_0601\ntrain_0602\ntrain_0603\ntrain_0604\ntrain_0605\ntrain_0606\ntrain_0607\ntrain_0608\ntrain_0609\ntrain_0610\ntrain_0611\ntrain_0612\ntrain_0613\ntrain_0614\ntrain_0615\ntrain_0616\ntrain_0617\ntrain_0619\ntrain_0620\ntrain_0622\ntrain_0623\ntrain_0626\ntrain_0627\ntrain_0629\ntrain_0630\ntrain_0631\ntrain_0632\ntrain_0633\ntrain_0634\ntrain_0635\ntrain_0636\ntrain_0637\ntrain_0638\ntrain_0639\ntrain_0640\ntrain_0641\ntrain_0642\ntrain_0643\ntrain_0645\ntrain_0647\ntrain_0648\ntrain_0651\ntrain_0652\ntrain_0653\ntrain_0654\ntrain_0655\ntrain_0657\ntrain_0660\ntrain_0665\ntrain_0666\ntrain_0668\ntrain_0669\ntrain_0671\ntrain_0672\ntrain_0673\ntrain_0674\ntrain_0675\ntrain_0676\ntrain_0677\ntrain_0678\ntrain_0679\ntrain_0680\ntrain_0681\ntrain_0682\ntrain_0683\ntrain_0684\ntrain_0685\ntrain_0686\ntrain_0687\ntrain_0688\ntrain_0690\ntrain_0691\ntrain_0692\ntrain_0693\ntrain_0695\ntrain_0696\ntrain_0697\ntrain_0698\ntrain_0699\ntrain_0700\ntrain_0701\ntrain_0702\ntrain_0703\ntrain_0704\ntrain_0705\ntrain_0706\ntrain_0707\ntrain_0708\ntrain_0709\ntrain_0710\ntrain_0711\ntrain_0712\ntrain_0713\ntrain_0714\ntrain_0715\ntrain_0716\ntrain_0717\ntrain_0720\ntrain_0721\ntrain_0724\ntrain_0725\ntrain_0728\ntrain_0730\ntrain_0731\ntrain_0732\ntrain_0734\ntrain_0735\ntrain_0736\ntrain_0738\ntrain_0739\ntrain_0740\ntrain_0741\ntrain_0743\ntrain_0744\ntrain_0745\ntrain_0746\ntrain_0747\ntrain_0748\ntrain_0749\ntrain_0751\ntrain_0753\ntrain_0755\ntrain_0757\ntrain_0760\ntrain_0761\ntrain_0765\ntrain_0766\ntrain_0767\ntrain_0769\ntrain_0771\ntrain_0773\ntrain_0774\ntrain_0775\ntrain_0776\ntrain_0778\ntrain_0780\ntrain_0781\ntrain_0782\ntrain_0783\ntrain_0784\ntrain_0785\ntrain_0786\ntrain_0788\ntrain_0789\ntrain_0791\ntrain_0792\ntrain_0793\ntrain_0794\ntrain_0795\ntrain_0796\ntrain_0797\ntrain_0798\ntrain_0799\ntrain_0801\ntrain_0802\ntrain_0803\ntrain_0804\ntrain_0806\ntrain_0807\ntrain_0808\ntrain_0809\ntrain_0810\ntrain_0811\ntrain_0812\ntrain_0813\ntrain_0816\ntrain_0817\ntrain_0821\ntrain_0822\ntrain_0823\ntrain_0824\ntrain_0825\ntrain_0826\ntrain_0827\ntrain_0828\ntrain_0829\ntrain_0830\ntrain_0831\ntrain_0832\ntrain_0833\ntrain_0835\ntrain_0836\ntrain_0837\ntrain_0838\ntrain_0839\ntrain_0840\ntrain_0841\ntrain_0842\ntrain_0843\ntrain_0846\ntrain_0847\ntrain_0848\ntrain_0849\ntrain_0850\ntrain_0851\ntrain_0852\ntrain_0853\ntrain_0854\ntrain_0855\ntrain_0856\ntrain_0858\ntrain_0859\ntrain_0860\ntrain_0861\ntrain_0862\ntrain_0863\ntrain_0865\ntrain_0866\ntrain_0868\ntrain_0869\ntrain_0871\ntrain_0872\ntrain_0873\ntrain_0874\ntrain_0876\ntrain_0877\ntrain_0878\ntrain_0879\ntrain_0880\ntrain_0881\ntrain_0882\ntrain_0883\ntrain_0884\ntrain_0885\ntrain_0886\ntrain_0887\ntrain_0888\ntrain_0889\ntrain_0890\ntrain_0891\ntrain_0892\ntrain_0893\ntrain_0894\ntrain_0895\ntrain_0896\ntrain_0897\ntrain_0898\ntrain_0899\ntrain_0900\ntrain_0901\ntrain_0902\ntrain_0903\ntrain_0904\ntrain_0905\ntrain_0906\ntrain_0907\ntrain_0908\ntrain_0909\ntrain_0910\ntrain_0911\ntrain_0912\ntrain_0913\ntrain_0915\ntrain_0919\ntrain_0920\ntrain_0921\ntrain_0922\ntrain_0925\ntrain_0926\ntrain_0929\ntrain_0930\ntrain_0931\ntrain_0932\ntrain_0934\ntrain_0936\ntrain_0937\ntrain_0938\ntrain_0939\ntrain_0940\ntrain_0942\ntrain_0944\ntrain_0946\ntrain_0949\ntrain_0950\ntrain_0951\ntrain_0952\ntrain_0955\ntrain_0956\ntrain_0957\ntrain_0961\ntrain_0962\ntrain_0963\ntrain_0964\ntrain_0965\ntrain_0966\ntrain_0967\ntrain_0968\ntrain_0969\ntrain_0970\ntrain_0971\ntrain_0972\ntrain_0973\ntrain_0974\ntrain_0975\ntrain_0976\ntrain_0979\ntrain_0980\ntrain_0981\ntrain_0982\ntrain_0983\ntrain_0984\ntrain_0986\ntrain_0988\ntrain_0990\ntrain_0992\ntrain_0993\ntrain_0994\ntrain_0995\ntrain_0996\ntrain_0997\ntrain_0998\ntrain_0999\ntrain_1002\ntrain_1003\ntrain_1004\ntrain_1005\ntrain_1006\ntrain_1009\ntrain_1010\ntrain_1011\ntrain_1012\ntrain_1013\ntrain_1015\ntrain_1017\ntrain_1020\ntrain_1022\ntrain_1023\ntrain_1024\ntrain_1025\ntrain_1026\ntrain_1027\ntrain_1028\ntrain_1029\ntrain_1030\ntrain_1031\ntrain_1032\ntrain_1033\ntrain_1034\ntrain_1035\ntrain_1036\ntrain_1038\ntrain_1039\ntrain_1040\ntrain_1041\ntrain_1044\ntrain_1046\ntrain_1050\ntrain_1052\ntrain_1053\ntrain_1054\ntrain_1055\ntrain_1056\ntrain_1057\ntrain_1058\ntrain_1059\ntrain_1061\ntrain_1063\ntrain_1064\ntrain_1066\ntrain_1067\ntrain_1068\ntrain_1071\ntrain_1072\ntrain_1074\ntrain_1075\ntrain_1076\ntrain_1077\ntrain_1078\ntrain_1079\ntrain_1080\ntrain_1082\ntrain_1083\ntrain_1084\ntrain_1085\ntrain_1086\ntrain_1087\ntrain_1088\ntrain_1090\ntrain_1093\ntrain_1094\ntrain_1095\ntrain_1098\ntrain_1099\ntrain_1100\ntrain_1103\ntrain_1104\ntrain_1105\ntrain_1106\ntrain_1107\ntrain_1110\ntrain_1111\ntrain_1113\ntrain_1114\ntrain_1115\ntrain_1116\ntrain_1119\ntrain_1121\ntrain_1122\ntrain_1125\ntrain_1127\ntrain_1130\ntrain_1131\ntrain_1132\ntrain_1133\ntrain_1134\ntrain_1135\ntrain_1136\ntrain_1138\ntrain_1139\ntrain_1140\ntrain_1141\ntrain_1142\ntrain_1143\ntrain_1144\ntrain_1145\ntrain_1146\ntrain_1149\ntrain_1150\ntrain_1151\ntrain_1153\ntrain_1154\ntrain_1156\ntrain_1157\ntrain_1158\ntrain_1159\ntrain_1161\ntrain_1162\ntrain_1164\ntrain_1165\ntrain_1166\ntrain_1167\ntrain_1168\ntrain_1169\ntrain_1171\ntrain_1172\ntrain_1173\ntrain_1174\ntrain_1176\ntrain_1177\ntrain_1178\ntrain_1179\ntrain_1181\ntrain_1183\ntrain_1184\ntrain_1185\ntrain_1186\ntrain_1187\ntrain_1191\ntrain_1192\ntrain_1193\ntrain_1194\ntrain_1195\ntrain_1196\ntrain_1197\ntrain_1199\ntrain_1200\ntrain_1202\ntrain_1203\ntrain_1204\ntrain_1205\ntrain_1207\ntrain_1208\ntrain_1210\ntrain_1211\ntrain_1212\ntrain_1213\ntrain_1214\ntrain_1217\ntrain_1218\ntrain_1219\ntrain_1221\ntrain_1222\ntrain_1223\ntrain_1224\ntrain_1226\ntrain_1227\ntrain_1228\ntrain_1229\ntrain_1231\ntrain_1236\ntrain_1237\ntrain_1238\ntrain_1240\ntrain_1241\ntrain_1242\ntrain_1244\ntrain_1245\ntrain_1247\ntrain_1250\ntrain_1252\ntrain_1254\ntrain_1255\ntrain_1256\ntrain_1257\ntrain_1259\ntrain_1260\ntrain_1261\ntrain_1262\ntrain_1263\ntrain_1264\ntrain_1266\ntrain_1268\ntrain_1270\ntrain_1271\ntrain_1274\ntrain_1275\ntrain_1276\ntrain_1277\ntrain_1279\ntrain_1280\ntrain_1281\ntrain_1282\ntrain_1283\ntrain_1284\ntrain_1285\ntrain_1286\ntrain_1287\ntrain_1290\ntrain_1291\ntrain_1292\ntrain_1293\ntrain_1294\ntrain_1296\ntrain_1297\ntrain_1299\ntrain_1301\ntrain_1302\ntrain_1303\ntrain_1306\ntrain_1308\ntrain_1312\ntrain_1313\ntrain_1314\ntrain_1315\ntrain_1316\ntrain_1318\ntrain_1319\ntrain_1320\ntrain_1322\ntrain_1323\ntrain_1324\ntrain_1326\ntrain_1327\ntrain_1328\ntrain_1329\ntrain_1330\ntrain_1331\ntrain_1332\ntrain_1333\ntrain_1334\ntrain_1335\ntrain_1336\ntrain_1337\ntrain_1338\ntrain_1339\ntrain_1340\ntrain_1341\ntrain_1343\ntrain_1344\ntrain_1345\ntrain_1347\ntrain_1348\ntrain_1349\ntrain_1354\ntrain_1357\ntrain_1358\ntrain_1359\ntrain_1360\ntrain_1361\ntrain_1362\ntrain_1364\ntrain_1365\ntrain_1366\ntrain_1367\ntrain_1368\ntrain_1369\ntrain_1370\ntrain_1371\ntrain_1372\ntrain_1374\ntrain_1375\ntrain_1376\ntrain_1377\ntrain_1378\ntrain_1381\ntrain_1382\ntrain_1385\ntrain_1386\ntrain_1387\ntrain_1388\ntrain_1389\ntrain_1390\ntrain_1391\ntrain_1392\ntrain_1394\ntrain_1395\ntrain_1396\ntrain_1397\ntrain_1398\ntrain_1400\ntrain_1401\ntrain_1402\ntrain_1403\ntrain_1404\ntrain_1406\ntrain_1407\ntrain_1409\ntrain_1411\ntrain_1412\ntrain_1413\ntrain_1414\ntrain_1415\ntrain_1417\ntrain_1418\ntrain_1419\ntrain_1422\ntrain_1423\ntrain_1424\ntrain_1425\ntrain_1426\ntrain_1427\ntrain_1428\ntrain_1429\ntrain_1430\ntrain_1431\ntrain_1434\ntrain_1435\ntrain_1436\ntrain_1438\ntrain_1439\ntrain_1441\ntrain_1442\ntrain_1445\ntrain_1446\ntrain_1447\ntrain_1449\ntrain_1450\ntrain_1451\ntrain_1452\ntrain_1453\ntrain_1454\ntrain_1457\ntrain_1462\ntrain_1463\ntrain_1464\ntrain_1469\ntrain_1477\ntrain_1479\ntrain_1480\ntrain_1481\ntrain_1482\ntrain_1483\ntrain_1484\ntrain_1485\ntrain_1487\ntrain_1489\ntrain_1490\ntrain_1493\ntrain_1494\ntrain_1495\ntrain_1496\ntrain_1498\ntrain_1499\ntrain_1500\ntrain_1502\ntrain_1503\ntrain_1504\ntrain_1505\ntrain_1507\ntrain_1508\ntrain_1509\ntrain_1510\ntrain_1511\ntrain_1513\ntrain_1514\ntrain_1515\ntrain_1516\ntrain_1517\ntrain_1518\ntrain_1520\ntrain_1521\ntrain_1522\ntrain_1523\ntrain_1524\ntrain_1527\ntrain_1529\ntrain_1531\ntrain_1534\ntrain_1537\ntrain_1538\ntrain_1541\ntrain_1543\ntrain_1545\ntrain_1547\ntrain_1548\ntrain_1549\ntrain_1550\ntrain_1551\ntrain_1552\ntrain_1553\ntrain_1555\ntrain_1558\ntrain_1559\ntrain_1561\ntrain_1562\ntrain_1563\ntrain_1564\ntrain_1565\ntrain_1566\ntrain_1567\ntrain_1568\ntrain_1569\ntrain_1570\ntrain_1573\ntrain_1575\ntrain_1576\ntrain_1577\ntrain_1578\ntrain_1579\ntrain_1580\ntrain_1581\ntrain_1582\ntrain_1583\ntrain_1584\ntrain_1585\ntrain_1587\ntrain_1588\ntrain_1590\ntrain_1591\ntrain_1592\ntrain_1593\ntrain_1594\ntrain_1595\ntrain_1597\ntrain_1599\ntrain_1600\ntrain_1602\ntrain_1603\ntrain_1604\ntrain_1605\ntrain_1607\ntrain_1609\ntrain_1610\ntrain_1611\ntrain_1612\ntrain_1616\ntrain_1618\ntrain_1620\ntrain_1621\ntrain_1622\ntrain_1624\ntrain_1626\ntrain_1628\ntrain_1629\ntrain_1630\ntrain_1631\ntrain_1632\ntrain_1633\ntrain_1634\ntrain_1635\ntrain_1636\ntrain_1638\ntrain_1641\ntrain_1642\ntrain_1643\ntrain_1644\ntrain_1645\ntrain_1646\ntrain_1647\ntrain_1648\ntrain_1649\ntrain_1650\ntrain_1651\ntrain_1652\ntrain_1655\ntrain_1657\ntrain_1659\ntrain_1660\ntrain_1662\ntrain_1663\ntrain_1664\ntrain_1665\ntrain_1667\ntrain_1669\ntrain_1670\ntrain_1672\ntrain_1673\ntrain_1674\ntrain_1675\ntrain_1676\ntrain_1677\ntrain_1678\ntrain_1679\ntrain_1681\ntrain_1682\ntrain_1683\ntrain_1685\ntrain_1686\ntrain_1687\ntrain_1688\ntrain_1689\ntrain_1691\ntrain_1692\ntrain_1694\ntrain_1695\ntrain_1696\ntrain_1698\ntrain_1700\ntrain_1701\ntrain_1704\ntrain_1706\ntrain_1709\ntrain_1711\ntrain_1714\ntrain_1716\ntrain_1717\ntrain_1720\ntrain_1721\ntrain_1722\ntrain_1723\ntrain_1725\ntrain_1728\ntrain_1729\ntrain_1730\ntrain_1733\ntrain_1734\ntrain_1735\ntrain_1736\ntrain_1738\ntrain_1739\ntrain_1740\ntrain_1741\ntrain_1742\ntrain_1743\ntrain_1748\ntrain_1749\ntrain_1750\ntrain_1751\ntrain_1753\n', 'umilife/completion_loss.py': '"""Memory-bounded causal CE on assistant tokens; Unsloth returns hidden states."""\nimport os\nimport torch\nimport torch.nn.functional as F\nfrom torch.utils.checkpoint import checkpoint\n\ndef completion_loss(model,batch,chunk_size=None):\n    chunk_size=chunk_size or int(os.environ.get("LIFEMATE_LOSS_CHUNK","256"))\n    assert chunk_size>0\n    # UNSLOTH_RETURN_HIDDEN_STATES=1 bypasses its fused vocabulary-sized loss allocation.\n    output=model(**batch,use_cache=False,return_dict=True)\n    hidden=output.logits\n    head=model.get_output_embeddings()\n    assert hidden.shape[-1]==head.weight.shape[1],\'Expected Unsloth hidden-state output, not vocabulary logits\'\n    labels=batch[\'labels\'][:,1:]\n    mask=labels!=-100\n    selected=hidden[:,:-1][mask];targets=labels[mask]\n    assert targets.numel()>0\n    def chunk_nll(h,t):\n        logits=F.linear(h.to(head.weight.dtype),head.weight,head.bias)\n        return F.cross_entropy(logits.float(),t,reduction=\'sum\')\n    total=hidden.new_zeros((),dtype=torch.float32)\n    for start in range(0,len(targets),chunk_size):\n        h=selected[start:start+chunk_size];t=targets[start:start+chunk_size]\n        if torch.is_grad_enabled() and h.requires_grad:\n            loss=checkpoint(chunk_nll,h,t,use_reentrant=False)\n        else:loss=chunk_nll(h,t)\n        total=total+loss\n    return total/targets.numel()\n', 'umilife/ddp_engine.py': '"""Explicit, symmetric DDP collectives; no Trainer monkey-patched control flow."""\nimport contextlib,hashlib,json,math,random,shutil,time,os,signal\nfrom pathlib import Path\nimport torch\nimport numpy as np\nimport torch.distributed as dist\nfrom torch.nn.parallel import DistributedDataParallel\n\n\ndef batches(lengths,epoch,rank,world,batch_size=4):\n    # Shuffle then length-bucket across both GPUs to reduce padding/stragglers.\n    indices=list(range(len(lengths)));random.Random(42+epoch).shuffle(indices)\n    width=batch_size*world\n    ordered=[]\n    for start in range(0,len(indices),width*32):\n        ordered.extend(sorted(indices[start:start+width*32],key=lambda i:lengths[i],reverse=True))\n    padding=(-len(ordered))%width\n    ordered.extend((ordered*math.ceil(padding/max(1,len(ordered))))[:padding])\n    return [ordered[i+rank*batch_size:i+(rank+1)*batch_size] for i in range(0,len(ordered),width)]\n\n\ndef state_digest(model):\n    h=hashlib.sha256()\n    for name,p in sorted(model.named_parameters()):\n        if p.requires_grad:\n            h.update(name.encode());h.update(p.detach().cpu().contiguous().view(torch.uint8).numpy().tobytes())\n    return h.hexdigest()\n\n\ndef agreement(value):\n    values=[None]*dist.get_world_size();dist.all_gather_object(values,value)\n    assert len(set(values))==1,values\n    return value\n\n\ndef weighted_eval(model,items,collate,device,amp,batch_size=4,loss_function=None):\n    # No DDP forward: rank workloads can differ by one validation example.\n    model.eval();rank=dist.get_rank();world=dist.get_world_size()\n    totals=torch.zeros(2,device=device,dtype=torch.float64)\n    local=list(range(rank,len(items),world))\n    with torch.inference_mode():\n        for start in range(0,len(local),batch_size):\n            x={k:v.to(device) for k,v in collate([items[i] for i in local[start:start+batch_size]]).items()}\n            n=(x[\'labels\'][:,1:]!=-100).sum()\n            with amp():loss=loss_function(model,x) if loss_function else model(**x,use_cache=False,return_dict=True).loss\n            totals[0]+=loss.double()*n;totals[1]+=n\n    dist.all_reduce(totals)\n    result=(totals[0]/totals[1]).item()\n    assert math.isfinite(result) and totals[1]>0,\'Nonfinite validation loss\'\n    model.train()\n    return result\n\n\nclass LossModule(torch.nn.Module):\n    # DDP static_graph must see the loss tensor directly even for custom ModelOutput objects.\n    def __init__(self,model,loss_function=None):super().__init__();self.model=model;self.loss_function=loss_function\n    def forward(self,**kwargs):\n        if self.loss_function:return self.loss_function(self.model,{k:v for k,v in kwargs.items() if k not in ("use_cache","return_dict")})\n        return self.model(**kwargs).loss\n\n\ndef fit(model,train,valid,collate,device,output,epochs,lr,save_model,load_model,\n        eval_steps=50,patience=3,min_delta=.001,amp_dtype=None,train_batch_size=4,eval_batch_size=4,loss_function=None,max_steps=None):\n    rank=dist.get_rank();world=dist.get_world_size();assert world==2\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    latest=output/\'latest.json\'\n    assert not (output/\'best\').exists() or latest.exists(),\'Incomplete legacy checkpoint cannot resume\'\n    params=[p for p in model.parameters() if p.requires_grad]\n    wrapped=DistributedDataParallel(LossModule(model,loss_function),device_ids=[device.index] if device.type==\'cuda\' else None,\n        broadcast_buffers=False,find_unused_parameters=False,static_graph=True)\n    amp=(lambda:torch.autocast(device_type=device.type,dtype=amp_dtype)) if amp_dtype else contextlib.nullcontext\n    scaler=torch.amp.GradScaler(\'cuda\',enabled=(device.type==\'cuda\' and amp_dtype==torch.float16),init_scale=1024)\n    optimizer=torch.optim.AdamW(params,lr=lr,weight_decay=.01)\n    lengths=[len(x[\'input_ids\']) for x in train]\n    steps_per_epoch=len(batches(lengths,0,rank,world,train_batch_size));total=steps_per_epoch*epochs\n    if max_steps is not None:total=min(total,max_steps)\n    warmup=max(1,math.ceil(.03*total))\n    def factor(step):\n        if step<warmup:return (step+1)/warmup\n        return .5*(1+math.cos(math.pi*min(1.,(step-warmup)/max(1,total-warmup))))\n    scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,factor)\n    initial=agreement(state_digest(model))\n    if rank==0:print(json.dumps({\'event\':\'initial_validation_start\',\'records\':len(valid)}),flush=True)\n    before=weighted_eval(model,valid,collate,device,amp,eval_batch_size,loss_function)\n    if rank==0:print(json.dumps({\'event\':\'initial_validation_end\',\'eval_loss\':before}),flush=True)\n    eval_seconds=0.;last_log=time.monotonic();last_log_step=0\n    best=math.inf;significant=math.inf;bad=0;best_step=0;history=[]\n    step=0;updates=0;skipped=0;stop=False;started=time.monotonic()\n    last_epoch=0.;saved_hash=None\n    resume_epoch=resume_batch=0\n    interrupted=[False]\n    signal.signal(signal.SIGTERM,lambda *_:interrupted.__setitem__(0,True))\n    signature=os.environ.get(\'TRAIN_RESUME_SIGNATURE\',\'\')\n    def persist(epoch,next_batch):\n        rng={\'numpy\':np.random.get_state(),\'python\':random.getstate(),\'cpu\':torch.get_rng_state(),\n             \'cuda\':torch.cuda.get_rng_state(device) if device.type==\'cuda\' else None}\n        rngs=[None]*world;dist.all_gather_object(rngs,rng)\n        dist.barrier()\n        if rank==0:\n            folder=output/f\'step-{step:08d}\';temp=output/\'saving.tmp\'\n            if temp.exists():shutil.rmtree(temp)\n            temp.mkdir();save_model(model,temp)\n            if (output/\'best\').exists():shutil.copytree(output/\'best\',temp/\'best_snapshot\')\n            torch.save({\'optimizer\':optimizer.state_dict(),\'scheduler\':scheduler.state_dict(),\n                \'scaler\':scaler.state_dict(),\'rng\':rngs,\'signature\':signature,\'world_size\':world,\n                \'epoch\':epoch,\'next_batch\':next_batch,\'step\':step,\'updates\':updates,\'skipped\':skipped,\n                \'best\':best,\'significant\':significant,\'bad\':bad,\'best_step\':best_step,\'history\':history,\n                \'saved_hash\':saved_hash,\'initial\':initial,\'before\':before,\'last_epoch\':last_epoch,\n                \'eval_seconds\':eval_seconds,\'total\':total},temp/\'training_state.pt\')\n            if folder.exists():shutil.rmtree(folder)\n            temp.rename(folder)\n            pointer=output/\'latest.tmp\';pointer.write_text(json.dumps({\'folder\':folder.name,\'step\':step}));pointer.replace(latest)\n            for old in sorted(output.glob(\'step-*\'))[:-2]:shutil.rmtree(old)\n        dist.barrier()\n    if latest.exists():\n        folder=output/json.loads(latest.read_text())[\'folder\']\n        state=torch.load(folder/\'training_state.pt\',map_location=\'cpu\',weights_only=False)\n        assert state[\'signature\']==signature and state[\'world_size\']==world and state[\'total\']==total,\'Resume settings mismatch\'\n        if rank==0:\n            if (output/\'best\').exists():shutil.rmtree(output/\'best\')\n            if (folder/\'best_snapshot\').exists():shutil.copytree(folder/\'best_snapshot\',output/\'best\')\n        dist.barrier()\n        load_model(model,folder);optimizer.load_state_dict(state[\'optimizer\']);scheduler.load_state_dict(state[\'scheduler\']);scaler.load_state_dict(state[\'scaler\'])\n        step=state[\'step\'];updates=state[\'updates\'];skipped=state[\'skipped\'];best=state[\'best\'];significant=state[\'significant\'];bad=state[\'bad\'];best_step=state[\'best_step\'];history=state[\'history\'];saved_hash=state[\'saved_hash\'];initial=state[\'initial\'];before=state[\'before\'];last_epoch=state[\'last_epoch\'];eval_seconds=state[\'eval_seconds\']\n        resume_epoch=state[\'epoch\'];resume_batch=state[\'next_batch\'];stop=bad>=patience\n        np.random.set_state(state[\'rng\'][rank][\'numpy\']);random.setstate(state[\'rng\'][rank][\'python\']);torch.set_rng_state(state[\'rng\'][rank][\'cpu\'])\n        if device.type==\'cuda\':torch.cuda.set_rng_state(state[\'rng\'][rank][\'cuda\'],device)\n        last_log_step=step\n        print(json.dumps({\'resumed_step\':step,\'stage_checkpoint\':str(folder)}),flush=True)\n    for epoch in range(epochs):\n        if epoch<resume_epoch:continue\n\n        local_batches=batches(lengths,epoch,rank,world,train_batch_size)\n        for batch_index,indices in enumerate(local_batches):\n            if epoch==resume_epoch and batch_index<resume_batch:continue\n            if step>=total or bad>=patience:break\n            model.train();optimizer.zero_grad(set_to_none=True)\n            x={k:v.to(device) for k,v in collate([train[i] for i in indices]).items()}\n            n=(x[\'labels\'][:,1:]!=-100).sum()\n            global_n=n.clone();dist.all_reduce(global_n)\n            with amp():\n                raw_loss=wrapped(**x,use_cache=False,return_dict=True)\n                # DDP averages rank gradients; recover true global token-mean gradient.\n                loss=raw_loss*(world*n/global_n)\n            finite=torch.isfinite(loss).to(torch.int32);dist.all_reduce(finite,op=dist.ReduceOp.MIN)\n            assert finite.item()==1,\'Nonfinite training loss on a rank\'\n            scaler.scale(loss).backward();scaler.unscale_(optimizer)\n            norm=torch.nn.utils.clip_grad_norm_(params,1.)\n            old_scale=scaler.get_scale();scaler.step(optimizer);scaler.update()\n            overflow=scaler.get_scale()<old_scale\n            flag=torch.tensor(int(overflow),device=device);dist.all_reduce(flag)\n            assert flag.item() in (0,world),"AMP overflow decisions diverged"\n            if overflow:skipped+=1\n            else:scheduler.step();updates+=1\n            step+=1;last_epoch=epoch+(batch_index+1)/len(local_batches)\n            if step%10==0 or step==1:\n                report={\'step\':step,\'planned_steps\':total,\'epoch\':last_epoch,\'local_loss\':raw_loss.item(),\n                        \'lr\':optimizer.param_groups[0][\'lr\'],\'grad_norm\':float(norm),\'optimizer_updates\':updates,\n                        \'seconds_per_step_including_eval\':(time.monotonic()-last_log)/(step-last_log_step)}\n                last_log=time.monotonic();last_log_step=step\n                if rank==0:print(json.dumps(report),flush=True)\n            at_end=step==total\n            if step%eval_steps==0 or at_end:\n                if rank==0:print(json.dumps({\'event\':\'validation_start\',\'step\':step,\'records\':len(valid)}),flush=True)\n                eval_started=time.monotonic()\n                metric=weighted_eval(model,valid,collate,device,amp,eval_batch_size,loss_function)\n                duration=time.monotonic()-eval_started;eval_seconds+=duration\n                improved=metric<best\n                if metric<significant-min_delta:significant=metric;bad=0\n                else:bad+=1\n                # Every rank computes the same scalar/state; only I/O is rank-zero-only.\n                if improved:\n                    best=metric;best_step=step\n                    dist.barrier()\n                    if rank==0:\n                        temp=output/\'best.tmp\'\n                        if temp.exists():shutil.rmtree(temp)\n                        temp.mkdir();save_model(model,temp)\n                        (temp/\'checkpoint.json\').write_text(json.dumps({\'step\':step,\'eval_loss\':metric,\'epoch\':last_epoch}))\n                        if (output/\'best\').exists():shutil.rmtree(output/\'best\')\n                        temp.rename(output/\'best\')\n                    dist.barrier()\n                    saved_hash=agreement(state_digest(model))\n                event={\'step\':step,\'epoch\':last_epoch,\'eval_loss\':metric,\'best_step\':best_step,\n                       \'best_loss\':best,\'patience_used\':bad,\'improved\':improved,\'evaluation_seconds\':duration}\n                history.append(event)\n                if rank==0:print(json.dumps(event),flush=True)\n                stop=bad>=patience\n                agreement(json.dumps([step,best,best_step,bad,stop]))\n            pause=torch.tensor(int(interrupted[0] or step==int(os.environ.get(\'SMOKE_PAUSE_STEP\',\'-1\')) or time.time()>=float(os.environ.get(\'TRAIN_DEADLINE\',\'inf\'))),device=device)\n            dist.all_reduce(pause,op=dist.ReduceOp.MAX)\n            if step%int(os.environ.get(\'SAVE_STEPS\',\'25\'))==0 or step%eval_steps==0 or at_end or stop or pause.item():\n                persist(epoch,batch_index+1)\n            if pause.item():raise SystemExit(75)\n            if stop or at_end:break\n        if stop or step>=total:break\n    assert updates>0 and best_step>0\n    dist.barrier();load_model(model,output/\'best\');dist.barrier()\n    final=agreement(state_digest(model));assert final==saved_hash and final!=initial\n    after=weighted_eval(model,valid,collate,device,amp,eval_batch_size,loss_function)\n    assert abs(after-best)<max(.005,abs(best)*.005),\'Best checkpoint restoration changed loss\'\n    result={\'world_size\':world,\'micro_batch\':train_batch_size,\'eval_batch_per_device\':eval_batch_size,\'gradient_accumulation\':1,\n            \'effective_batch\':train_batch_size*world,\'epochs\':epochs,\'actual_epoch\':last_epoch,\'global_step\':step,\n            \'optimizer_updates\':updates,\'amp_skipped_updates\':skipped,\'planned_steps\':total,\n            \'learning_rate\':lr,\'scheduler\':\'cosine\',\'warmup_ratio\':.03,\n            \'early_stopping\':{\'patience\':patience,\'min_delta\':min_delta,\'eval_steps\':eval_steps,\n                \'restore_best\':True,\'best_step\':best_step,\'best_metric\':best,\'stopped_early\':stop},\n            \'before\':{\'eval_loss\':before},\'after\':{\'eval_loss\':after},\'history\':history,\n            \'initial_trainable_sha256\':initial,\'final_trainable_sha256\':final,\n            \'wall_s\':time.monotonic()-started,\'train_padding_examples_per_epoch\':(-len(train))%(train_batch_size*world),\n            \'validation_repeats\':0,\'scheduled_evaluation_seconds\':eval_seconds}\n    dist.barrier();del wrapped\n    return result\n', 'umilife/english_labels.py': '"""Concise field-source plans. Deterministic labels, not teacher-model reasoning."""\nimport re\nVERSION=\'concise-field-sources-v2\'\n\ndef leaves(value):\n    if isinstance(value,dict):\n        for v in value.values():yield from leaves(v)\n    elif isinstance(value,list):\n        for v in value:yield from leaves(v)\n    else:yield value\n\ndef contains(value,target):\n    return target is not None and target!=\'\' and any(type(v) is type(target) and v==target for v in leaves(value))\n\ndef rationale(row,action,trajectory):\n    name,args=action[\'name\'],action[\'arguments\']\n    if name==\'sys.ask_missing_info\':\n        return (\'Request clarification for \'+args[\'tool_name\']+\': \'+\', \'.join(args[\'params\'])+\n                \'. Wait for the missing information before performing the incomplete operation.\')\n    # Gold values determine visible source attribution only; never serialize them into thought.\n    groups={};history=row.get(\'history\',[])\n    for field,value in args.items():\n        source=None\n        for prior in reversed(trajectory):\n            if contains(prior[\'observation\'],value):\n                source=\'the \'+prior[\'action\'][\'name\']+\' result\';break\n        if source is None and contains(history,value):source=\'conversation history\'\n        if source is None:source=\'the current request and context\'\n        groups.setdefault(source,[]).append(field)\n    thought=\'Perform the next pending \'+name.replace(\'_\',\' \').replace(\'.\',\' / \')+\' operation.\'\n    if groups:\n        bindings=\'; \'.join(\'resolve \'+\', \'.join(fields)+\' from \'+source for source,fields in groups.items())\n        thought+=\' \'+bindings[0].upper()+bindings[1:]+\'.\'\n    else:thought+=\' Use the result to advance the pending request.\'\n    if re.search(r\'\\b(nếu|khi|trên|dưới)\\b\',row[\'audio_text\'].lower()):\n        thought+=" Respect the user\'s condition using available evidence before any dependent write."\n    return thought\n\ndef terminal(row,trajectory):\n    if not trajectory:\n        noop=bool(re.search(r\'thôi|khỏi|không cần|đừng|cảm ơn|khen|chào\',row[\'audio_text\'].lower()))\n        return ({\'thought\':\'No tool operation is needed; end this turn without changing state.\' if noop else\n                 \'The request cannot be completed with the supported tools; stop without inventing an operation.\',\n                 \'next\':\'summarizer\' if noop else \'give_up\'},\'No changes were made.\' if noop else None)\n    if trajectory[-1][\'action\'][\'name\']==\'sys.ask_missing_info\':\n        return {\'thought\':\'Clarification was requested; end this turn and wait for the user.\',\'next\':\'summarizer\'},\'Please provide the requested missing information.\'\n    # Preserve annotated terminal routes without claiming every observation confirms success.\n    return {\'thought\':\'End this turn using the available results; do not repeat completed operations.\',\'next\':\'summarizer\'},\'The recorded tool sequence has ended.\'\n', 'umilife/preflight_2gpu.py': '"""Executed by the user before Qwen loading: imports, NCCL and AMP backward on both GPUs."""\nimport os,json\nimport torch\nlocal=int(os.environ[\'LOCAL_RANK\']);torch.cuda.set_device(local)\nfrom unsloth import FastLanguageModel\nimport torch.distributed as dist\nfrom datetime import timedelta\nfrom torch.nn.parallel import DistributedDataParallel\nfrom umilife.ddp_engine import agreement\n\ndef main():\n    assert torch.cuda.device_count()==2 and int(os.environ[\'WORLD_SIZE\'])==2\n    dist.init_process_group(\'nccl\',device_id=torch.device(\'cuda\',local),timeout=timedelta(minutes=5))\n    dtype=torch.bfloat16 if torch.cuda.get_device_capability(local)[0]>=8 else torch.float16\n    model=torch.nn.Linear(16,8).to(torch.device(\'cuda\',local));wrapped=DistributedDataParallel(model,device_ids=[local],static_graph=True)\n    optimizer=torch.optim.AdamW(model.parameters(),lr=.001)\n    scaler=torch.amp.GradScaler(\'cuda\',enabled=dtype==torch.float16,init_scale=1024)\n    for _ in range(2):\n        optimizer.zero_grad(set_to_none=True)\n        with torch.autocast(\'cuda\',dtype=dtype):loss=wrapped(torch.randn(4,16,device=torch.device(\'cuda\',local))).float().square().mean()\n        scaler.scale(loss).backward();scaler.step(optimizer);scaler.update()\n    assert all(torch.isfinite(p).all() for p in model.parameters())\n    x=torch.tensor([1.],device=torch.device(\'cuda\',local));dist.all_reduce(x);assert x.item()==2\n    agreement(\'NCCL_AMP_BATCH4_PASS\')\n    if dist.get_rank()==0:print(json.dumps({\'status\':\'pass\',\'world_size\':2,\'micro_batch\':4,\'backend\':\'nccl\',\'amp\':str(dtype)}),flush=True)\n    dist.barrier();dist.destroy_process_group()\nif __name__==\'__main__\':main()\n', 'umilife/prepare_2gpu.py': '"""Fixed episode split, concise field-source plans, prior-evidence-only inputs."""\nimport argparse,hashlib,json\nfrom copy import deepcopy\nfrom collections import Counter\nfrom pathlib import Path\nfrom jsonschema import Draft202012Validator\nfrom umilife.core import read_jsonl,write_json,dumps,digest,allowed_specs,load_catalog,import_toolenv\nfrom umilife.data import annotation_tasks,normalized_row\nfrom umilife.prompt_policy import PROMPTS,build_messages,request_schema,POLICY_AUDIT\nfrom umilife.train import encode\nfrom umilife.english_labels import rationale,terminal as grounded_terminal,VERSION\n\nRATIONAL_SHA=\'d14768df60e155521a57998a5c6d0c4c1cc3b6372b74cce7a955098fd3fdf497\'\nSPLIT_SHA=\'631f8cfab61ddde833477f743cf1a65a9012526e0178e97046a36ee864e3d505\'\nTRAIN_SHA=\'24c42c79fbe8db932fe9817d041c0d0ca422ac8177b413cc1be0cda1fd8941a8\'\n\ndef main():\n    p=argparse.ArgumentParser()\n    for name in (\'data-root\',\'output\',\'split-file\',\'model\',\'revision\'):p.add_argument(\'--\'+name,required=True)\n    a=p.parse_args();root=Path(a.data_root);out=Path(a.output);out.mkdir(parents=True,exist_ok=True)\n    assert digest(root/\'train/train.jsonl\')==TRAIN_SHA\n    from transformers import AutoTokenizer\n    tokenizer=AutoTokenizer.from_pretrained(a.model,revision=a.revision)\n    ids=Path(a.split_file).read_text().splitlines();valid=set(ids);assert len(ids)==len(valid)==245\n    rows=read_jsonl(root/\'train/train.jsonl\');assert len(rows)==1545\n    assignments={r[\'id\']:\'validation\' if r[\'id\'] in valid else \'train\' for r in rows}\n    assert valid=={k for k,v in assignments.items() if v==\'validation\'}\n    catalog=load_catalog(root,\'train\');ToolEnv=import_toolenv(root,\'train\')\n    counts=Counter();references=[];token_stats={};schema_cache={}\n    handles={(role,part):(out/f\'{role}.{part}.jsonl\').open(\'w\') for role in (\'global\',\'planner\',\'caller\',\'summarizer\') for part in (\'train\',\'validation\')}\n    def add(role,row,specs,trajectory,target,plan=None):\n        part=assignments[row[\'id\']]\n        schema=request_schema(role,specs,plan);key=dumps(schema)\n        if key not in schema_cache:schema_cache[key]=Draft202012Validator(schema)\n        schema_cache[key].validate(target)\n        rec={\'id\':row[\'id\'],\'role\':role,\'step\':len(trajectory),\n             \'messages\':build_messages(role,normalized_row(row),row[\'audio_text\'],specs,trajectory,plan),\n             \'target\':dumps(target)}\n        encoded=encode(tokenizer,rec);n=len(encoded[\'input_ids\']);assert n<=8192,(\'Overlength\',row[\'id\'],role,n)\n        stat=token_stats.setdefault(f\'{role}.{part}\',{\'records\':0,\'max_tokens\':0,\'max_target_tokens\':0})\n        stat[\'records\']+=1;stat[\'max_tokens\']=max(stat[\'max_tokens\'],n)\n        stat[\'max_target_tokens\']=max(stat[\'max_target_tokens\'],sum(x!=-100 for x in encoded[\'labels\']))\n        handles[role,part].write(dumps(rec)+\'\\n\');counts[f\'{role}.{part}\']+=1\n    try:\n        for row in rows:\n            specs=allowed_specs(row,catalog);env=ToolEnv(\'train\',row[\'id\'],root=root);trajectory=[]\n            gold=row[\'label\'][\'tool_calls\'];results=row[\'label\'][\'tool_results\'];assert len(gold)==len(results)\n            for step,(action,expected) in enumerate(zip(gold,results,strict=True)):\n                assert env.validate(action[\'name\'],action[\'arguments\']).get(\'ok\')\n                observed=env.call(action[\'name\'],action[\'arguments\']);assert observed==expected,(row[\'id\'],step)\n                thought=rationale(row,action,trajectory)\n                plan={\'thought\':thought,\'next\':\'caller\',\'tool\':action[\'name\']}\n                add(\'planner\',row,specs,trajectory,plan)\n                add(\'caller\',row,specs,trajectory,action,plan)\n                add(\'global\',row,specs,trajectory,{**plan,\'action\':action})\n                trajectory.append({\'thought\':thought,\'action\':action,\'observation\':observed})\n            references.append({\'id\':row[\'id\'],\'tool_calls\':gold,\'state_hash\':env.get_state_hash()})\n            plan,answer=grounded_terminal(row,trajectory)\n            add(\'planner\',row,specs,trajectory,plan)\n            add(\'global\',row,specs,trajectory,{**plan,**({\'answer\':answer} if answer is not None else {})})\n            if answer is not None:add(\'summarizer\',row,specs,trajectory,{\'answer\':answer},plan)\n    finally:\n        for f in handles.values():f.close()\n    manifest={\'split\':assignments,\'counts\':dict(counts),\'excluded\':[],\n        \'train_episodes\':1300,\'validation_episodes\':245,\'split_sha256\':digest(a.split_file),\'train_sha256\':TRAIN_SHA,\n        \'model\':a.model,\'revision\':a.revision,\'rationale_authors\':[\'Codex\'],\'rationales_sha256\':digest(Path(__file__).with_name(\'english_labels.py\')),\n        \'rationale_provenance\':\'English evidence-compiled labels \'+VERSION+\'; deterministic annotation rules, not independent per-example human/LLM review.\',\n        \'summary_labels\':\'Deterministic concise facts from final gold tool observation; max 180 answer tokens; not human-written summaries.\',\n        \'zero_action_convention\':\'Cancellation/acknowledgement heuristic -> summarizer; other zero-call requests -> give_up. Audit ambiguous cases before claiming semantic correctness.\',\n        \'validation_usage\':\'Held-out labels only for loss evaluation and post-generation scoring; never optimizer updates or inference prompts.\',\n        \'training_transcripts\':\'Provided clean audio_text; post-training evaluation uses supplied ASR with no oracle fallback.\',\n        \'prompts_sha256\':POLICY_AUDIT[\'static_prompts_sha256\'],\'policy\':POLICY_AUDIT,\n        \'tokenization\':token_stats,\'files_sha256\':{f.name:digest(f) for f in out.glob(\'*.jsonl\')}}\n    write_json(out/\'references.json\',references);write_json(out/\'manifest.json\',manifest)\n    print(dumps(manifest),flush=True)\nif __name__==\'__main__\':main()\n', 'umilife/prompt_policy.py': '"""Generic tool policy. No benchmark examples, held-out labels or per-sample rules."""\nfrom .typescript_schema import typescript_catalog\nfrom copy import deepcopy\nfrom datetime import datetime,timedelta\nimport hashlib,json\n\nPOLICY_VERSION=\'planner-caller-concise-ts-v2\'\nCOMMON="""You are LifeMate, a personal assistant using installed tools.\nAct only on the current request. History resolves references, not proof of execution in this turn. Preserve quantities, negation, conditions and exact text to send or save. Use the supplied clock and timezone. Never invent identifiers or tool results.\nOnly current observations confirm execution. Do not repeat successful effects, even with reworded arguments. Retry only with evidence of failure and a justified correction. Reuse unchanged read results; change strategy after an unhelpful lookup.\nTool contracts are TypeScript declarations: ? marks optional fields, literal unions specify enums, and comments describe constraints and identifier sources. These are interface documents, not executable code. Return one JSON object without markdown.\n"""\nPROMPTS={\n \'planner\':COMMON+"""You are Planner. Decide the next single step toward pending user objectives.\nChoose the installed tool, order dependent operations, evaluate conditions from available evidence, and stop when no further authorized operation is needed. If a condition or required identifier needs discovery, select an appropriate read tool first. Choose sys.ask_missing_info when required information cannot be resolved from context or supported lookups. An unresolved ID alone is not an unsupported request.\nWrite thought in concise English: normally one or two sentences, targeting at most 60 words. State the objective, essential selection constraints, and where Caller should obtain values. Refer to the request/history or a specific previous tool result. Caller resolves records, identifiers, field values and date formatting. Do not serialize the complete arguments or narrate all previous steps. Retain names, times or other user-supplied selectors when needed to distinguish pending objectives. Explain the decision rather than merely restating a tool name.\nFor an operation return {thought,next:"caller",tool}. For a completed/no-op turn or after requesting clarification return {thought,next:"summarizer"}; this terminal route does not require a trained Summarizer. For an unsupported request return {thought,next:"give_up"}. Never stop while a supported, authorized objective remains, repeat a confirmed write for a reply, or claim unconfirmed success.\n""",\n \'caller\':COMMON+"""You are Caller. Convert Planner\'s selected step into one valid API call.\nReturn exactly {name,arguments}; name must equal plan.tool. Planner owns routing and tool selection. Resolve values using the current request, history, context and previous tool observations. Treat plan.thought as guidance, not proof. Match the intended entity before taking its ID; use identifiers actually observed or explicitly supplied. Respect user qualifiers rather than choosing the first result. Normalize dates with the supplied clock/timezone and preserve exact text to send or save.\nRead the selected TypeScript contract, comments, required/optional fields, enums and alternative argument groups. Supply only supported values; omit optional fields without evidence. For sys.ask_missing_info, use the intended tool_name and exact missing parameter names from its contract. Never invent a required identifier or change the selected tool. Correct validation feedback without repeating confirmed effects. Output no thought, next or answer. The runtime executes your JSON.\n""",\n \'summarizer\':COMMON+"Return only {answer}, briefly grounded in actual observations. Do not claim unconfirmed success.",\n \'global\':COMMON+"Handle one ReAct step: return {thought,next,tool,action} for caller, {thought,next,answer} for summarizer, or {thought,next} for give_up. Keep the English thought concise; action contains {name,arguments}."\n}\nCOMPLETION_REVIEW="""Review this proposed ending against every objective in the current request. Has something only been looked up but not executed? Are you confusing history with current execution? Are there unfinished objectives, actual operation failures, or unresolved empty searches? Do not mistake business-state fields for execution failures. Do not propose repeating any confirmed effect just to verify it or generate a reply; summarize when all objectives are satisfied. A different argument spelling does not justify repeating a completed objective. Continue with caller when work remains. Prefer lookup before clarification. Summarize only with sufficient evidence or while waiting for clarification; give_up only beyond installed capabilities."""\n\ndef canonical(value):return json.dumps(value,ensure_ascii=False,sort_keys=True,separators=(\',\',\':\'))\ndef digest(value):return hashlib.sha256(canonical(value).encode()).hexdigest()\ndef obj(properties,required):return {\'type\':\'object\',\'properties\':properties,\'required\':required,\'additionalProperties\':False}\n\ndef parameter_schema(schema):\n    """Convert documented x-one-of to ordinary JSON Schema; preserve typed nesting."""\n    allowed={\'type\',\'properties\',\'required\',\'items\',\'enum\',\'format\',\'minimum\',\'maximum\',\n             \'exclusiveMinimum\',\'exclusiveMaximum\',\'minLength\',\'maxLength\',\'minItems\',\'maxItems\',\n             \'anyOf\',\'oneOf\',\'allOf\',\'additionalProperties\',\'description\'}\n    out={}\n    for key,value in schema.items():\n        if key not in allowed:continue\n        if key==\'properties\':out[key]={name:parameter_schema(prop) for name,prop in value.items()}\n        elif key==\'items\' and isinstance(value,dict):out[key]=parameter_schema(value)\n        elif key in (\'anyOf\',\'oneOf\',\'allOf\'):out[key]=[parameter_schema(x) for x in value]\n        else:out[key]=deepcopy(value)\n    if out.get(\'type\')==\'object\' and \'properties\' in out:out[\'additionalProperties\']=False\n    if schema.get(\'x-one-of\'):\n        # Full object alternatives are supported by both JSON Schema and LMFE.\n        base=out\n        out={\'anyOf\':[{**deepcopy(base),\'required\':list(dict.fromkeys(base.get(\'required\',[])+list(group)))}\n                      for group in schema[\'x-one-of\']]}\n    return out\n\ndef action_schema(specs,name):\n    arguments=parameter_schema(specs[name][\'parameters\'])\n    if name==\'sys.ask_missing_info\':\n        # Only public installed schemas provide these choices, never an answer label.\n        choices=[n for n,s in specs.items() if n!=name and s[\'parameters\'].get(\'properties\')]\n        if choices:arguments[\'properties\'][\'tool_name\'][\'enum\']=sorted(choices)\n        arguments[\'properties\'][\'params\'][\'minItems\']=1\n        fields=sorted({key for target in choices for key in specs[target][\'parameters\'][\'properties\']})\n        if fields:arguments[\'properties\'][\'params\'][\'items\'][\'enum\']=fields\n    return obj({\'name\':{\'type\':\'string\',\'enum\':[name]},\'arguments\':arguments},[\'name\',\'arguments\'])\n\ndef route_schema(specs,global_mode=False):\n    thought={\'type\':\'string\',\'maxLength\':1024};branches=[]\n    if specs:\n        if global_mode:\n            for name in sorted(specs):\n                branches.append(obj({\'thought\':thought,\'next\':{\'type\':\'string\',\'enum\':[\'caller\']},\n                    \'tool\':{\'type\':\'string\',\'enum\':[name]},\'action\':action_schema(specs,name)},[\'thought\',\'next\',\'tool\',\'action\']))\n        else:\n            branches.append(obj({\'thought\':thought,\'next\':{\'type\':\'string\',\'enum\':[\'caller\']},\n                \'tool\':{\'type\':\'string\',\'enum\':sorted(specs)}},[\'thought\',\'next\',\'tool\']))\n    done={\'thought\':thought,\'next\':{\'type\':\'string\',\'enum\':[\'summarizer\']}}\n    if global_mode:done[\'answer\']={\'type\':\'string\',\'maxLength\':1024}\n    branches.append(obj(done,list(done)))\n    branches.append(obj({\'thought\':thought,\'next\':{\'type\':\'string\',\'enum\':[\'give_up\']}},[\'thought\',\'next\']))\n    return {\'anyOf\':branches}\n\ndef compact_catalog(specs):\n    fields=(\'type\',\'enum\',\'x-ref\',\'format\',\'items\',\'minimum\',\'maximum\')\n    return [{\'name\':name,\'kind\':spec.get(\'kind\'),\'description\':spec[\'description\'],\n             \'parameters\':{key:{k:deepcopy(v) for k,v in prop.items() if k in fields}\n                           for key,prop in spec[\'parameters\'].get(\'properties\',{}).items()},\n             \'required\':spec[\'parameters\'].get(\'required\',[]),\n             **({\'one_of\':spec[\'parameters\'][\'x-one-of\']} if spec[\'parameters\'].get(\'x-one-of\') else {})}\n            for name,spec in sorted(specs.items())]\n\ndef reference_clock(context):\n    result={}\n    try:\n        now=datetime.fromisoformat(context[\'current_time\'])\n        result={\'today\':now.date().isoformat(),\'tomorrow\':(now+timedelta(days=1)).date().isoformat(),\n                \'day_after_tomorrow\':(now+timedelta(days=2)).date().isoformat(),\n                \'weekday_iso\':now.isoweekday(),\'timezone\':context.get(\'timezone\')}\n    except (KeyError,TypeError,ValueError):pass\n    return result\n\ndef build_messages(role,row,text,specs,trajectory,plan=None,feedback=None):\n    # Explicit provenance allowlist. Current labels/final state/reference transcript never enter here.\n    context={k:deepcopy(row[\'context\'][k]) for k in (\'current_time\',\'timezone\',\'location\',\'user_profile\') if k in row[\'context\']}\n    history=[{k:deepcopy(turn[k]) for k in (\'role\',\'text\',\'tool_calls\',\'tool_results\') if k in turn}\n             for turn in row.get(\'history\',[])]\n    content={\'context\':context,\'reference_clock\':reference_clock(context),\'history\':history,\n             \'request\':text,\'trajectory\':deepcopy(trajectory)}\n    if role in (\'planner\',\'global\'):content[\'tools_ts\']=typescript_catalog(specs)\n    elif role==\'caller\':\n        selected=(plan or {}).get(\'tool\');assert selected in specs,\'Planner selected an unavailable tool\'\n        content.update(plan=deepcopy(plan),selected_tool_ts=typescript_catalog({selected:specs[selected]},detailed=True))\n        if selected==\'sys.ask_missing_info\':\n            content[\'missing_information_tools_ts\']=typescript_catalog({name:spec for name,spec in specs.items()\n                if name!=selected and spec[\'parameters\'].get(\'properties\')})\n    else:content[\'plan\']=deepcopy(plan)\n    if feedback is not None:content[\'validation_feedback\']=deepcopy(feedback)\n    # Keep the original request at the end, after schemas/observations/feedback.\n    content.pop(\'request\');content[\'request\']=text\n    return [{\'role\':\'system\',\'content\':PROMPTS[role]},\n            {\'role\':\'user\',\'content\':json.dumps(content,ensure_ascii=False,separators=(\',\',\':\'))}]\n\ndef request_schema(role,specs,plan=None):\n    if role==\'caller\':return action_schema(specs,plan[\'tool\'])\n    if role in (\'planner\',\'global\'):return route_schema(specs,role==\'global\')\n    return obj({\'answer\':{\'type\':\'string\',\'maxLength\':1024}},[\'answer\'])\n\ndef validate_meta_action(action,specs):\n    if action[\'name\']!=\'sys.ask_missing_info\':return None\n    args=action[\'arguments\'];target=args.get(\'tool_name\');params=args.get(\'params\',[])\n    if target not in specs or target==\'sys.ask_missing_info\':return \'Clarification target must be another installed tool\'\n    # An optional API parameter can still be necessary to express the user\'s intent.\n    fields=set(specs[target][\'parameters\'].get(\'properties\',{}))\n    if not params or len(params)!=len(set(params)) or not set(params)<=fields:\n        return \'params must contain unique parameter names from the target tool contract\'\n    return None\n\ndef repeated_read(action,specs,trajectory):\n    if specs[action[\'name\']].get(\'kind\')!=\'read\':return False\n    for old in reversed(trajectory):\n        if specs.get(old[\'action\'][\'name\'],{}).get(\'kind\')==\'write\':return False\n        if canonical(old[\'action\'])==canonical(action):return True\n    return False\n\nPOLICY_AUDIT={\'policy_version\':POLICY_VERSION,\'static_prompts_sha256\':digest(PROMPTS),\n    \'completion_review_sha256\':digest(COMPLETION_REVIEW),\'few_shot_examples\':0,\n    \'sample_specific_rules\':False,\'gold_inputs_to_model\':False,\n    \'allowed_input_sources\':[\'current ASR request\',\'ASR user history\',\'provided context\',\n        \'installed public tool schemas\',\'observations from executed tools\',\'model-generated plan/feedback\']}\n', 'umilife/train_2gpu.py': '"""Notebook runtime: two-rank Unsloth QLoRA, explicit DDP loop and best adapter."""\nimport os\nos.environ.setdefault(\'UNSLOTH_COMPILE_DISABLE\',\'0\')\nos.environ.setdefault(\'UNSLOTH_RETURN_LOGITS\',\'0\')\nos.environ[\'UNSLOTH_RETURN_HIDDEN_STATES\']=\'1\'\nos.environ.setdefault(\'TOKENIZERS_PARALLELISM\',\'false\')\n\ndef main():\n    import argparse,json,hashlib,importlib.metadata\n    from pathlib import Path\n    import torch\n    local=int(os.environ[\'LOCAL_RANK\']);torch.cuda.set_device(local)\n    os.environ.setdefault(\'TORCHINDUCTOR_CACHE_DIR\',f\'/tmp/lifemate_inductor_rank{local}\')\n    os.environ.setdefault(\'TRITON_CACHE_DIR\',f\'/tmp/lifemate_triton_rank{local}\')\n    print(json.dumps({\'stage\':\'startup\',\'rank\':local,\'unsloth_compile_enabled\':os.environ[\'UNSLOTH_COMPILE_DISABLE\']==\'0\'}),flush=True)\n    from unsloth import FastLanguageModel\n    import torch.distributed as dist\n    from datetime import timedelta\n    from peft import set_peft_model_state_dict\n    from peft.utils.save_and_load import load_peft_weights\n    from umilife.core import digest,write_json,dumps\n    from umilife.models import TARGET_MODULES,seed_all\n    from umilife.train import Dataset,Collator,adapter_state\n    from umilife.prompt_policy import PROMPTS,POLICY_AUDIT\n    from umilife.ddp_engine import fit,agreement\n    from umilife.training_modes import validate_initialization,verify_parent\n    from umilife.completion_loss import completion_loss\n    p=argparse.ArgumentParser()\n    for k in (\'stage\',\'data-dir\',\'output-dir\',\'model\',\'revision\'):p.add_argument(\'--\'+k,required=True)\n    p.add_argument(\'--train-batch-size\',type=int,default=4);p.add_argument(\'--eval-batch-size\',type=int,default=4)\n    p.add_argument(\'--smoke\',action=\'store_true\');p.add_argument(\'--eval-steps\',type=int,default=100)\n    p.add_argument(\'--init-from\',choices=(\'base\',\'global\'),required=True)\n    p.add_argument(\'--global-adapter\');p.add_argument(\'--epochs\',type=int,required=True)\n    a=p.parse_args()\n    assert a.train_batch_size>0 and a.eval_batch_size>0\n    assert int(os.environ[\'WORLD_SIZE\'])==2 and torch.cuda.device_count()==2\n    assert a.stage in (\'global\',\'planner\',\'caller\',\'summarizer\')\n    assert a.epochs==1 if a.stage==\'global\' else a.epochs>0\n    validate_initialization(a.stage,a.init_from,a.global_adapter)\n    dist.init_process_group(\'nccl\',timeout=timedelta(minutes=20),device_id=torch.device(\'cuda\',local))\n    seed_all(42)\n    data=Path(a.data_dir);out=Path(a.output_dir);target=out/\'adapters\'/a.stage\n    assert not (target/\'lineage.json\').exists(),\'Stage already complete\'\n    if target.exists():\n        import shutil;shutil.rmtree(target)\n    manifest=json.loads((data/\'manifest.json\').read_text())\n    prompt_sha=POLICY_AUDIT[\'static_prompts_sha256\']\n    assert manifest[\'train_episodes\']==1300 and manifest[\'validation_episodes\']==245\n    assert manifest[\'prompts_sha256\']==prompt_sha and manifest[\'rationale_authors\']==[\'Codex\']\n    for part in (\'train\',\'validation\'):\n        f=data/f\'{a.stage}.{part}.jsonl\';assert digest(f)==manifest[\'files_sha256\'][f.name]\n    dtype=torch.bfloat16 if torch.cuda.get_device_capability(local)[0]>=8 else torch.float16\n    model,tokenizer=FastLanguageModel.from_pretrained(model_name=a.model,revision=a.revision,\n        use_exact_model_name=True,max_seq_length=8192,dtype=dtype,load_in_4bit=True,\n        fast_inference=False,device_map={\'\':local},trust_remote_code=False)\n    model=FastLanguageModel.get_peft_model(model,r=16,lora_alpha=32,lora_dropout=0,\n        target_modules=TARGET_MODULES,bias=\'none\',use_gradient_checkpointing=True,random_state=42)\n    parent=None\n    def load_adapter(model,path):\n        result=set_peft_model_state_dict(model,load_peft_weights(str(path),device=\'cpu\'))\n        assert not result.unexpected_keys,result.unexpected_keys\n    if a.global_adapter:\n        parent=Path(a.global_adapter)\n        lineage=verify_parent(parent,a.model,a.revision,digest(data/\'manifest.json\'),prompt_sha,a.smoke)\n        load_adapter(model,parent)\n    initial=adapter_state(model)\n    if parent:assert initial[\'sha256\']==lineage[\'final_adapter\'][\'sha256\'],\'Global initialization mismatch\'\n    assert all(\'lora_\' in n for n,v in model.named_parameters() if v.requires_grad)\n    FastLanguageModel.for_training(model)\n    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={\'use_reentrant\':False})\n    model.config.use_cache=False\n    def cached_dataset(part):\n        path=data/f\'{a.stage}.{part}.jsonl\'\n        cache=data/f\'{a.stage}.{part}.{digest(path)[:16]}.tokens.pt\'\n        if dist.get_rank()==0 and not cache.exists():\n            encoded=Dataset(path,tokenizer,8192)\n            temp=cache.with_suffix(\'.tmp\');torch.save({\'items\':encoded.items,\'stats\':encoded.stats},temp);temp.replace(cache)\n        dist.barrier()\n        payload=torch.load(cache,map_location=\'cpu\',weights_only=True)\n        encoded=Dataset.__new__(Dataset);encoded.items=payload[\'items\'];encoded.stats=payload[\'stats\']\n        return encoded\n    train=cached_dataset(\'train\');valid=cached_dataset(\'validation\')\n    assert train.stats[\'overlength\']==valid.stats[\'overlength\']==0,\'No training examples may be silently dropped\'\n    if a.smoke:\n        train.items=sorted(train.items,key=lambda x:len(x[\'input_ids\']),reverse=True)[:16]\n        valid.items=sorted(valid.items,key=lambda x:len(x[\'input_ids\']),reverse=True)[:8]\n        assert len(train.items)==16 and len(valid.items)==8\n    os.environ[\'TRAIN_RESUME_SIGNATURE\']=hashlib.sha256(json.dumps({\'manifest\':digest(data/\'manifest.json\'),\'stage\':a.stage,\'epochs\':a.epochs,\'smoke\':a.smoke,\'batch\':a.train_batch_size,\'eval_batch\':a.eval_batch_size,\'eval_steps\':a.eval_steps,\'compile\':os.environ.get(\'UNSLOTH_COMPILE_DISABLE\'),\'loss_chunk\':os.environ.get(\'LIFEMATE_LOSS_CHUNK\',\'256\'),\'model\':a.model,\'revision\':a.revision,\'init_from\':a.init_from,\'parent\':digest(parent/\'adapter_model.safetensors\') if parent else None},sort_keys=True).encode()).hexdigest()\n    def save_adapter(model,path):model.save_pretrained(path,safe_serialization=True)\n    result=fit(model,train,valid,Collator(tokenizer.pad_token_id),torch.device(\'cuda\',local),\n        out/\'checkpoints\'/a.stage,a.epochs,5e-5 if a.stage==\'global\' else 1e-5,\n        save_adapter,load_adapter,amp_dtype=dtype,train_batch_size=a.train_batch_size,\n        eval_batch_size=a.eval_batch_size,loss_function=completion_loss,\n        eval_steps=1 if a.smoke else a.eval_steps,max_steps=2 if a.smoke else None)\n    final=adapter_state(model);assert final[\'finite\'] and final[\'sha256\']!=initial[\'sha256\']\n    agreement(final[\'sha256\'])\n    if dist.get_rank()==0:\n        save_adapter(model,target);tokenizer.save_pretrained(target)\n        history=result.pop(\'history\');write_json(target/\'training_history.json\',history)\n        record={**result,\'stage\':a.stage,\'model\':a.model,\'revision\':a.revision,\n            \'initialization\':a.init_from,\'method\':(\'Unsloth QLoRA global\' if a.stage==\'global\' else \'Unsloth QLoRA local-from-\'+a.init_from),\'loss_implementation\':\'assistant-only chunked CE, checkpointed projection\',\'loss_chunk\':int(os.environ.get(\'LIFEMATE_LOSS_CHUNK\',\'256\')),\'smoke\':a.smoke,\'initial_adapter\':initial,\'final_adapter\':final,\n            \'parent\':str(parent) if parent else None,\n            \'global_adapter_sha256\':digest(parent/\'adapter_model.safetensors\') if parent else None,\n            \'dataset_manifest_sha256\':digest(data/\'manifest.json\'),\'prompt_sha256\':prompt_sha,\n            \'tokenization\':{\'train\':train.stats,\'validation\':valid.stats},\n            \'actual_records\':{\'train\':len(train),\'validation\':len(valid)},\n            \'peak_cuda_gb_rank0\':torch.cuda.max_memory_allocated()/2**30,\n            \'unsloth_compile_enabled\':os.environ.get(\'UNSLOTH_COMPILE_DISABLE\')==\'0\',\n            \'versions\':{k:importlib.metadata.version(k) for k in (\'torch\',\'transformers\',\'unsloth\',\'unsloth_zoo\',\'peft\',\'bitsandbytes\')}}\n        write_json(target/\'lineage.json\',record);print(dumps(record),flush=True)\n    dist.barrier();dist.destroy_process_group()\nif __name__==\'__main__\':main()\n', 'umilife/training_modes.py': '"""Pure configuration/lineage checks, independent of Torch and GPUs."""\nimport json\nfrom pathlib import Path\n\nLOCAL_STAGES = (\'planner\', \'caller\', \'summarizer\')\n\ndef stages_for(mode, selected=LOCAL_STAGES):\n    assert mode in (\'global\', \'local\', \'global_to_local\'), \'Unknown TRAIN_MODE\'\n    assert selected and len(set(selected)) == len(selected) and set(selected) <= set(LOCAL_STAGES)\n    if mode == \'global\': return (\'global\',)\n    if mode == \'local\': return tuple(selected)\n    return (\'global\', *selected)\n\ndef validate_initialization(stage, init_from, global_adapter):\n    assert stage in (\'global\', *LOCAL_STAGES)\n    assert init_from in (\'base\', \'global\')\n    if stage == \'global\': assert init_from == \'base\' and not global_adapter\n    assert bool(global_adapter) == (init_from == \'global\'), \'Global parent is required only for init-from=global\'\n    return init_from\n\ndef parent_for(stage, mode, local_init, external_parent, output):\n    assert local_init in (\'base\', \'global\')\n    if stage == \'global\': return None\n    if mode == \'global_to_local\': return Path(output)/\'adapters/global\'\n    if local_init == \'base\':\n        assert external_parent is None, \'Remove GLOBAL_ADAPTER when LOCAL_INIT=base\'\n        return None\n    assert external_parent is not None, \'Set GLOBAL_ADAPTER for LOCAL_INIT=global\'\n    return Path(external_parent)\n\ndef verify_parent(path, model, revision, manifest_sha, prompt_sha, smoke):\n    path = Path(path)\n    meta = json.loads((path/\'lineage.json\').read_text())\n    assert meta[\'stage\'] == \'global\' and meta[\'model\'] == model and meta[\'revision\'] == revision\n    assert meta[\'dataset_manifest_sha256\'] == manifest_sha, \'Parent Global was trained with a different data split/labels\'\n    assert meta[\'prompt_sha256\'] == prompt_sha and meta[\'final_adapter\'][\'finite\']\n    assert smoke or not meta.get(\'smoke\', False), \'Full training cannot initialize from a smoke adapter\'\n    assert (path/\'adapter_model.safetensors\').is_file()\n    return meta\n', 'umilife/typescript_schema.py': '"""Compact TypeScript interfaces derived solely from the installed public catalog."""\nimport json\n\ndef literal(v):return json.dumps(v,ensure_ascii=False,separators=(\',\',\':\'))\ndef ts_type(p):\n    if \'enum\' in p:return \' | \'.join(literal(v) for v in p[\'enum\'])\n    t=p.get(\'type\')\n    if isinstance(t,list):return \' | \'.join(ts_type({**p,\'type\':v}) for v in t)\n    if t==\'array\':return \'(\'+ts_type(p.get(\'items\',{}))+\')[]\'\n    if t==\'object\':\n        fields=p.get(\'properties\')\n        if fields is None:return \'Record<string, unknown>\'\n        base=\'{ \'+ \'; \'.join(literal(k)+(\'\' if k in p.get(\'required\',[]) else \'?\')+\': \'+ts_type(v) for k,v in fields.items())+\' }\'\n        if p.get(\'x-one-of\'):\n            alt=[\'{ \'+\'; \'.join(literal(k)+\': \'+ts_type(fields[k]) for k in group)+\' }\' for group in p[\'x-one-of\']]\n            return base+\' & (\'+\' | \'.join(alt)+\')\'\n        return base\n    return {\'string\':\'string\',\'integer\':\'number\',\'number\':\'number\',\'boolean\':\'boolean\',\'null\':\'null\'}.get(t,\'unknown\')\n\ndef metadata(p,path=\'\',detailed=False):\n    result=[]\n    for k,v in p.items():\n        if k in (\'properties\',\'items\',\'required\',\'type\',\'enum\',\'x-one-of\'):continue\n        if k==\'description\' and not detailed:continue\n        result.append((path+\'.\' if path else \'\')+k+\'=\'+literal(v))\n    if p.get(\'type\')==\'integer\':result.append(path+\':integer\')\n    for k,v in p.get(\'properties\',{}).items():result.extend(metadata(v,(path+\'.\' if path else \'\')+k,detailed))\n    if p.get(\'items\'):result.extend(metadata(p[\'items\'],path+\'[]\',detailed))\n    return result\n\ndef typescript_catalog(specs,detailed=False):\n    lines=[\'interface InstalledTools {\']\n    for name,s in sorted(specs.items()):\n        comment=\'; \'.join([s.get(\'kind\',\'\'),s.get(\'description\',\'\'),*metadata(s[\'parameters\'],detailed=detailed)])\n        if s.get(\'returns\') and detailed:comment+=\'; returns=\'+str(s[\'returns\'])\n        comment=comment.replace(\'*/\',\'* /\').replace(\'\\n\',\' \')\n        lines.append(\'  /** \'+comment+\' */\')\n        lines.append(\'  \'+literal(name)+\'(args: \'+ts_type(s[\'parameters\'])+\'): unknown;\')\n    return \'\\n\'.join(lines+[\'}\'])\n'}
SOURCE_SHA='a8a46bda154b372a1d0d7e837bc9a8a1be4aa879a60b390e609d1d6ee0db0c3c'
assert hashlib.sha256(json.dumps(PAYLOAD,sort_keys=True).encode()).hexdigest()==SOURCE_SHA
for name,content in PAYLOAD.items():
 p=SOURCE/name;p.parent.mkdir(parents=True,exist_ok=True);p.write_text(content)
config={'source_sha':SOURCE_SHA,'model':MODEL,'revision':REVISION,'epochs':EPOCHS,'micro_batch':4,'world_size':2,'train_mode':TRAIN_MODE,'local_init':LOCAL_INIT if TRAIN_MODE=='local' else None,'compile_enabled':True,'local_stages':list(LOCAL_STAGES),'global_parent_sha':hashlib.sha256((Path(GLOBAL_ADAPTER)/'adapter_model.safetensors').read_bytes()).hexdigest() if GLOBAL_ADAPTER else None}
if (WORK/'run_config.json').exists():assert json.loads((WORK/'run_config.json').read_text())==config,'Incompatible resume directory'
else:(WORK/'run_config.json').write_text(json.dumps(config,indent=2))


In [ ]:
sys.path.insert(0,str(SOURCE))
from umilife.paths import find_data_root
DATA_ROOT,archive=find_data_root('/kaggle/input')
if DATA_ROOT is None:
 assert archive is not None,'Attach the competition dataset'
 dest=Path('/tmp/english_evidence_data');dest.mkdir(exist_ok=True)
 with zipfile.ZipFile(archive) as z:
  assert all((dest/m.filename).resolve().is_relative_to(dest.resolve()) for m in z.infolist())
  z.extractall(dest)
 DATA_ROOT,_=find_data_root(dest)
subprocess.run([sys.executable,'-m','pip','install','--quiet','uv==0.9.10'],check=True)
UV=[sys.executable,'-m','uv'];os.environ['UV_NATIVE_TLS']='true'
venv=Path('/tmp/english_evidence_unsloth');PYTHON=str(venv/'bin/python')
if not Path(PYTHON).exists():subprocess.run([*UV,'venv','--python','3.12','--seed',str(venv)],check=True)
subprocess.run([*UV,'pip','sync','--index-strategy','unsafe-best-match','--python',PYTHON,str(SOURCE/'requirements_train_2gpu.lock')],check=True)
subprocess.run([*UV,'pip','check','--python',PYTHON],check=True)
ENV=os.environ.copy()
for key in ('PYTHONPATH','PYTHONHOME'):ENV.pop(key,None)
ENV.update(CUDA_VISIBLE_DEVICES='0,1',PYTHONNOUSERSITE='1',TOKENIZERS_PARALLELISM='false',OMP_NUM_THREADS='2',
 UNSLOTH_COMPILE_DISABLE='0',LIFEMATE_LOSS_CHUNK='256',SAVE_STEPS='25',TRAIN_DEADLINE=str(DEADLINE),HF_HOME='/tmp/lifemate_hf_cache')
purelib=Path(subprocess.check_output([PYTHON,'-I','-c',"import sysconfig;print(sysconfig.get_path('purelib'))"],text=True).strip())
ENV['LD_LIBRARY_PATH']=':'.join(str(p) for p in (purelib/'nvidia').glob('*/lib'))+':'+ENV.get('LD_LIBRARY_PATH','')
def execute(module,args,ddp=False,env=None,check=True):
 command=[PYTHON,'-u','-m']+(['torch.distributed.run','--standalone','--nproc_per_node=2','--module',module] if ddp else [module])
 return subprocess.run(command+list(map(str,args)),cwd=SOURCE,env=env or ENV,check=check)
# No synthetic GPU preflight or model smoke. Verify real training startup in the log.
for key in ('SMOKE_PAUSE_STEP',):ENV.pop(key,None)
DATA=WORK/'prepared'
if not (DATA/'manifest.json').exists():
 execute('umilife.prepare_2gpu',['--data-root',DATA_ROOT,'--output',DATA,'--split-file',SOURCE/'split.txt','--model',MODEL,'--revision',REVISION])
m=json.loads((DATA/'manifest.json').read_text());assert m['train_episodes']==1300 and m['validation_episodes']==245
assert m['split_sha256']==hashlib.sha256((SOURCE/'split.txt').read_bytes()).hexdigest()
print('Data:',m['counts']);print('Annotations:',m['rationale_provenance'])


In [ ]:
from umilife.training_modes import stages_for,parent_for,verify_parent
SELECTED_STAGES=stages_for(TRAIN_MODE,LOCAL_STAGES)
print('Selected stages:',SELECTED_STAGES,'Local initialization:',LOCAL_INIT if TRAIN_MODE=='local' else TRAIN_MODE)
def train_profile(profile):
 output=WORK/profile
 for stage in SELECTED_STAGES:
  epochs=EPOCHS[stage]
  parent=parent_for(stage,TRAIN_MODE,LOCAL_INIT,GLOBAL_ADAPTER,output)
  init_from='global' if parent is not None else 'base'
  if parent is not None:
   verify_parent(parent,MODEL,REVISION,hashlib.sha256((DATA/'manifest.json').read_bytes()).hexdigest(),m['prompts_sha256'],False)
  done=output/'adapters'/stage/'lineage.json'
  if done.exists():
   record=json.loads(done.read_text())
   assert record['dataset_manifest_sha256']==hashlib.sha256((DATA/'manifest.json').read_bytes()).hexdigest()
   assert record['smoke']==(False) and record['initialization']==init_from
   assert record['global_adapter_sha256']==(hashlib.sha256((parent/'adapter_model.safetensors').read_bytes()).hexdigest() if parent else None)
   print('Already complete:',profile,stage);continue
  args=['--stage',stage,'--data-dir',DATA,'--output-dir',output,'--model',MODEL,'--revision',REVISION,
   '--init-from',init_from,'--epochs',epochs,'--train-batch-size',4,'--eval-batch-size',4,'--eval-steps',100]
  stage_env=ENV.copy()
  if parent is not None:args+=['--global-adapter',parent]
  result=execute('umilife.train_2gpu',args,ddp=True,env=stage_env,check=False)
  assert result.returncode==0,'Training paused or failed. Preserve WORK and inspect logs/latest checkpoint before resuming.'
 print('Completed:',profile)
train_profile('full')
print('Preserve this directory:',WORK)
